# Kaggriculture X594: Ryo-live adaptive closed-loop planner

This notebook emits the standalone X594 `main.py`, based on the complete Ryo Hasegawa replay archive. It preserves the observed opening package and uses a closed-loop planner that adapts to public board, market, and inventory state.

## Evidence and scope

X594 is an experiment against the current X578 control. The local paired gate was negative, so this notebook version is for controlled submission testing rather than a claim that X594 replaces X578.


In [ ]:
from __future__ import annotations

import base64
import hashlib
from pathlib import Path

EXPECTED_SHA256 = "d946545e01d5a07485ca348c16080b61ecd4f2eff6a8c5fc08be7a449c43abc5"
SOURCE_B85 = (
    "A|fJKH90gO;((MOY-x67En{J9X<~9=bY)~9VPs)&bZK^FAYo@^ZgegR3RGxmb0BkcVQyq$Y;SI5AYo@^Zge1Zb7gZNbZBKDaCKsA"
    "X=5N`Y;SXAWG!rOZ*U-RY+-J0WpW^4a${&|bY)|7b#i4OVs&&1ZDDj{Xk~LCbZBKDZ*XO9X>MmAZ(?(0a&~28AZczOVQg$5F*G?K"
    "WN&wFY;R#?Wn>^yd2b+cWpHd^d2>1-W@&b1AZTfFWpgYFba!tcb7*B{a4aBncW)qLZ+CMnAZ2W2c4clLZDnk4ZgVUkb7^=WcW7l{"
    "bRctOWn^<KAYpD~AZBlMav*nTWnpw6W@Tk$E(%m=Wgu)}bY*fNWMyM%b7^mGb0BhMZDDC{Aa7!GWpZ|5bZKvHEo5?Oc4clLa$$66"
    "WpW^NXkl(3a%FIAVR<cdVQ^(+E(#(dA_@v3A|gv+XJ>M0V|8qFb#i4OVP|D-bRaD)AY*KAb7f>LY;SLHAaHD9Zf<3AASiBcAaZ4J"
    "Y+-pIbYXC1DJ}{MQ*?4+bY*9GAaiwXZDDeG3N0-yEiElAEiElAEiElAEea|iL}7U#FghSKAaiJCWpE&GZXk4MWgv8EY-MvGZe@6M"
    "AarjabZBKDb7*B`ASiclZ)_kdAZBH9bZKm9dS!ATVRB_4bZBJ?ARuL7a%_1ZV_|b>AZ2c6X>MgHEFdv3AZ=xAZ*FrSZ*CxTXk{Q~"
    "VR9f&S0Hq0Y-MvGC}eeQaAjm5Z*CxDVR;}iFexkwARu8NY-x0KY-J#KXk}q^E+8O5ZfR{{Y;z!UX>=fIZXjVGV{dJ6VPkY4dT(xJ"
    "AYpQEb#7!JbZBKDb7*B`AaidZbZBJ?ARuI6X>55QW@Tk$ATJ<eVRB_4FCb%YY;0v?bRaJvXkl`8Wpi{Oa&L8RWFTpCAaiJMa&#^V"
    "Dj-8|cXJ?Na%CW4WMpMzAZBuJZ6IV}c_1_(VRIm3VRL98VRCY5c4c!QC^#Tuc_4mdVR;}pEFflOcV%)QX=WgFXk{RDZ+C7AARumK"
    "c4cxPZ*XO9b0A?LZE0+4AaiJMa49<=b7*B{a3FJIVQgg}cWHEJAbDYOZXk1XZ*paGE($6jQ*?4+cVcC7a%p9AAYpQ4AaHD9Zggd2"
    "AZ1~4Y<VCkW^!+BAY@^AATul=els>LG%_G!c_3tAc_1+{DJ&pnWpZ?BY-xIBWC|c4Z*CxAXJvCBIUr$fWFRp!ASh&Sbz*E~b0B4Q"
    "Wpa5Sb7N>_WOZz1WFT;IZ)A02bZKvHDIj5PWFTl^a&~2NbY)~9bZBJ?ARujTZDnqBAZBuPX>=fIb0BYSAarPDAaHD9ZgegnAXjK*"
    "VRRs7X>4qBAZ2!Ca(Q%UX>MmAWo&b0AYpD~AZc?Ta%FIAVQzF~WC|c4b#8QNY#?M|c_1=3J0NCdWn>^}b0A`Gb!TXFAa`hGZXk4M"
    "Wguo@a%~`Kb0BkQZ*p`l3MwE>VQyqPAWlUfVRCPEZe$>2VR;}mEjBD5Q&%8iW^`q8AarPDAZ=xAZ*CxMZ*FCIE+8O5AaiA7Z*F8D"
    "ZDnk4ZXkDIc4Z)Ga|$3JaBN|2bY)~9Z*CxDVR;}iF(7YlY<VDQW*~HEWgu;3Y;SHLZDDe2Wpp5Eb0BVSbRcJJb#!!PWG)ISAVqd%"
    "a(N(hb#iVXVIXR6Vjyg3b95kSb0A`MX>4>LW^!+BAarPDAa7!GWpZ|9WFTf?a%~`UbYXO5AYpD~Aa!nObaM(IASh;Ga&2XDAS)nf"
    "VQyq|DIj5TWgu-~bYo~`WFT~JAZl-7b0A`QAa!zQWo~16AarSEax5Tla%pdJX>@rkAY^HCbYX5|Wgua0WC|c4b98BAYiVv}b8|Z&"
    "b#7^Nb0B7EZfSF9AarPDAa`$aYanlKAarPDAarSLWgv8DWqBZTbYX5}Aa8CUVr6D;a%CWGZ+2;JXD$jVAWdO%Yh`pgAZBH9bZKm9"
    "dS!ATFCcAYY;SHLFCcJoWo>D7Z6IfFZ)9^IVRB_4b8l>9AYpSLb8l~MAYpSLbZBLHAaZ44V`vH>AarPDAaiJCWFRPaXm4e9WpW^M"
    "Wo&G7AZBTDb95kQWpr~ObZBKDaB^v5WhpEmcW7l{bRc47d2eoHAarPDAZBG{WC|c4a%FR6a&~1PX>%ZRZ){{NAYpD~AZ2!Ca(Q%U"
    "X>MmAX>%ZKX>oOFWMOn=WFTUBAaitOa3D7^I4%kzA|eWDZE$aLbRcbEbZ81`ZE$aLbRchY3JMBjWo95>Yi@61C~jeGWh@|MWoBV@"
    "Y;-9)3LqdLAarthItm~lARr(hARuyObairWAar?fWhi82W?^+~bSWrrb1r3Wc4=~NZZ2nKbSQ3NZDlMVWMyVyb!>DgDGDGUARuLU"
    "V`Xr3AVqj%WpH$9Z*DpYARr(hARr(ha%FUNa&91GWoBV@Y;+0=BOom;EiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElA"
    "EiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiDQoAVqFxX>Mg8V{dMAbYX6Eb08>fX>xLJa&sVSVP|J-WnX1(c4=~N"
    "Zf#|5baO9dZgz7oYhh<+a%p3AY;<*UWhn|HAT2E|EiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElA"
    "EiElAEiElAEiElAEiElAEiElAEiElAEeb<YPf$}JJs^7uARr(hB3DR7K~y3-ARr(hARv1pb7f^@B03;3Ff1S<W@&PBbRs$+GAtk>"
    "ZDDw1VR<4tAT%r>B57`PWpZ|5Y$7@!Ff1S<ZDDv{d1+;AWFk5sHY^|_Z*FIAX>MmCIv_@2Y;$FOED9hXARr<`K~hprR3bVcARr(h"
    "dm?jXWn>~cATls4AR=aIa&vSdIv_GEAR=vHcw}LDB03;5EFdCjZggdGc42HHIv_ACAR=vHcwc#GWo%?3Iv_MGAR=#WXK!h4XCgWv"
    "MqzAoWqm9PARr(hB2-UJK~zs7Iv^k*ARv1pb7f^@B03;7Ff1S<W@&PBbRs$+I4mF{ZDDw1VR<4tAUG@_B57`PWpZ|5Y$7@!F)Sb="
    "ZDDv{d1+;AWFk5sG%O$@Z*FIAX>MmCIv`YXb!B}l3LqdLAR<##QbAWjMN(2(B03;@B6DSBWFk5sF)%PJAR=aIa&vSdIv_DHEFdCn"
    "VR&R=c_KO>F)%D3B57`PWpZ|5Y$7@!GAtk>ZDDv{d1+;AWFk5sG%O$@Z*FIAX>MmCIv`YXb!B}l3LqdLAR<jgOixZCIv^k*ARr)n"
    "B6DSBWFk5sI4~?AB4%lFb95p)ATcm3AR=vHcw}LDB03;3GAtk>X>N37a&}>CB03;2EFdCnVR&D8X=Q9=B03;8EFdCpZf9?4Zf7Do"
    "AVy(qb7g%j3VjMePDxEcOj95|AbScRARr(jM^8^vMIt&Ndm>|Rb95p)ATuy9EFdCtbaHiLbairNB03-<Lr+goA}k;xW@&PBbRs$+"
    "G%O$@X>N37a&}>CB03;3EFdCnVR&C?Wo%?3Iv_MGAR=&bZ)A02bRs$+B1K0>B7H0hARr(hB12DCB03--AbTQXZ*z1aIv_MKFf1S<"
    "b98cbV{~<LWg<EtB2Yn7R8>+%A}k;xW@&PBbRs$+I4mF{X>N37a&}>CB03;4EFdCnVR&C?Wo%?3Iv_SIAR=&bZ)A02bRs$+B27t5"
    "OCo(N3LqdLAR<#pMMY2|Iv{%@V{dbGB03;7Ffc42B6D<dbz^jOa%CbqAR<sfQ&d$_MItO9B4%lFb95p)AT}%@B57`PWpZ|5Y$7@!"
    "Gb|t?ZDDv{Xk~0<B03;8EFdCqa&KgHV{{@qAR<>!PfQ|xEDC)JP*P7sRYO!$AUz;kB3DR7K~y3vAR<FSQc_P;A}k;xR8LJoR8Jx-"
    "AR<##QbAWjMN(2(A}k;xO+`#kP9iKIB1K0>A}k;xO-W2kA}k;xS5Hq&A}k;xMnzIoNlZyvMN%SN3Q$r-O-WTvAUz-`B27t5OCl^F"
    "B3DmOOd>2GB2!dSL03XWQc_tWEFdCHMNCglA}I<@K~hUaR9{IjAUz;4FfcGM3Qa*$OGQ*)P(e~bO;aE}AbScRARr(jS4c%cR3bVc"
    "ARr(hAbTQWVRL07Iv_GNEFdCOB03;6Ffc42B4TDDIv^r*adLDbEFdCcbRs$+FfKSSEFdCbW+FNuB5ZGGA}k;xVRRxoATTa6Fnufv"
    "ARr(hB11t^QcqMOIv^k*ARv1pVqtS-B03;5H7p<^R3bVcG&L|RAR=OBB03-<XlZU|Wg;vfB4TtRIv_DFFfc42B4K7CIv^r*adLDb"
    "EFdCbbRs$+FfKPReJlzfARr(jR8LJoR8Jy0ARr(hAbTQWVRL07Iv_SMEFdCOB03;4Ffc42B4TDDIv^rwX>Mm_A}k;xVss)pATTa8"
    "Ff1S<VP+yaAR=>da&#grAR=LOB03;2E;cZIED9hXARr=BR8m1#LPb(iSt2?hdm>_Cb7dkrATcs9EFdCOB03;3Ffc42B4TDDIv^r*"
    "adLDbEFdCcbRs$+FfKPREFdCbW+FNuB5Y}HWnpq6EFdCbbRs$+F)lVReJlzfARr(jO+`#kP9i!WARr(hAbTQWVRL07Iv_GNFf1S<"
    "R3bVcGcYhLAR=OBB03-<Y;R{GEFdCcbRs$+FfKAMEFdCbW+FNuB6D#fEFdCbbRs$+GcGnTeJlzfARr(jMMp;>Iv^k*ARr(hAbTQW"
    "VRL07Iv_PLEFdCOB03;5Gcqh7B4TDDIv^rwX>Mm_A}k;xVss)pATTa8Ff1S<VP+yaAR=sUXCf>hB4KnQIv_AEGBABC3LqdLAR<jk"
    "OiLm<ARr(hARr)nB4S~4Wg<EtF*YzPAR<&EIv_DJGAtk>VrC*bAR=>da&#grAR=OPB03;2E;cYMAR=LAB03-<Y-w&~VR9lYAR=LO"
    "B03;3E;cZIED9hXARr=FPftuDIv^k*ARr(hdm>_Cb7dkrATls8EFdCOB03;3Ff}Y7B4TDDIv^r!Z)YMbAR=OPB03;2E;2AIAR=LA"
    "B03-<b8#XpAR=LOB03;5E;2BEED9hXARr<}MN(8rOi5ZrQX)Dadm>_Cb7dkrATcm7EFdCOB03;4Ffc42B4TDDIv^r!X>Mg<aw04s"
    "B4TtRIv_AEG%ze6B4K7CIv^r!X>Mg<aw04sB4KnQIv_AEG%$TE3VjMwNKa5xAUz;^3LqdLAR<CROGQ#yB03;jB1K0>A}k;xS4c%c"
    "R3cq03LqdLAR<snT3SJ0Q%FxxB03;jB27t5OCl^FB2-UJK~zs7EFdCRNJT+ZB3&#BARr(hB0^GCPD4mvQ&3M-B03;jB1K0>A}k;x"
    "S4c%cR3a=OB2!dSL03XWQc_tWT`US9ARr(jSwT`xUsF_1Qbi&<AX_3=PftuDT`US9ARr(jNkc_nLsCUSO<z+;Pf#K{AX_3+R8m1#"
    "LPb(iSt2YTB27t5OCl^FB3DR7K~y4LED9hXARr=8MO0ryK}JO)Iv`sjLqSqfPgEjZED9hXARr=BO;1l$NJ&LsQ%FxxB03;jB2!dS"
    "L03XWQc_tWEFdCHNlZ&3T`US9ARr(jMnO_dMN(5=O+iviMN}d>AX_3=NJT+ZA}k;xLqSqfPgEi-AR<&xO+i#oA}k;xQ&dtxS3*Tn"
    "QduHhEDC)JOhHaWUr$m*MN%L=AX_3%MItO9B2!l)EFdCNMIv1aOhHaWUr<s>Lq$^{Js?{#FfcGIATls8Ff1T6FfcG(3PMjoQbZs<"
    "ATclsR8T}9Js>hP3QR#$R9{3vSs*<iGC2xNK~q#;Q&dGzAUz;AF*qO~ARr(hARr(hBOq*Hb95kcbY*ZLcW7^OWguZ=bZKvHb0A@I"
    "WgumEWn*=8Wn>CdNJT_nLqSj=Js>eKFbY#hMMPgzNlZmkAUz;kC^Re}G$||~C^ak~G$||~C^Re}H7P70C^ak~H7Q*RLq$$hMN%L="
    "ASg60H7p=BE;T6%3JPRpW*}d4Xkl<=C}wqTV=N$eEFe@VItm~lARu@kJs@picqlL~Ff1T=DGDGUARuXGAZB%LV<0^}AR=sOZe?L|"
    "B035nARr(hARr)eWps6NZXkFHARr(hX=Wg1b#7xIJv|^Ib8#X%3LqdLARr(hAaZ4Nb#iVXcpxeucnTmOARuXGAZB%LV<0^}AR=>d"
    "a&#g(3LqdLARr(hAaZ4Nb#iVXZDDk1E^~2mbSQW!3LqdLAZcbGW_503AU!=GB5ZGGB035nARr(hARr)eWps6NZXj)8bZ9PYZ)Yem"
    "E-)Y~Ab2SXARr(hX=Wg1b#7xIJv|^IXlZU|Wg<EXARr(hARr(hbs#+;cpxtzR0<#<ARr(hARuyObairWAax)sAUG~CASxhjVR$Go"
    "E-)-0bs#MuF)lDEASx;#G72CdARuyObairWAb1K23S?zwAZ=lCYh`p_aB^v5WhiNMWo;}VX>N8YItm~lARurcJs?d%QcFctUr<3("
    "K}}OzX>?_6T?!x|ARuC4b7d?bR3JSda9bi`VRL07T`VAQTOw2<T?!x|ARuXGAZc!PAUq&VK~hUaR9{IjItm~lARr(hARu9Fa3DP("
    "a9bi`bRu0KDj;HEb7deeAYXH6VQ^(Aa9bi`W+GiIAXF?MR4ED|ARr(hARr)ca%p2_AUz;rVRL05D<ENQa3CrmUvp?-aAhcPTOwj+"
    "B3&#XO+iviMO0r&Fd!`;X>N8bAXF&|ARr(hWo&b0Itm~lARr(hARu9Fa3DP(a9bi_bRu0KDj;HEb7deeAYXH6VQ^(Aa9bi_W+GiI"
    "AXF?MR4ED|ARr(hARr)ca%p2_AUz;rVRL05Eg)fSa3CrmUvp?-aAhcPTOwg*B3&#XX>N8PEg(%nQcFctUr8`5AXF&|ARr(ha%FUN"
    "a&91PVR$GpEFfuabSQFfb#7!RaB^v5Whp5s3JMBjWo96CZfSILUtwZzc4c31a%p2_C~0(MZ7d*ZZgwmnW^8Y7a$j(AX=7z9AZ=lI"
    "Uv+M2baN>>3LqdLAR;0nNN;x_ZDDSCAa!nObaNnMVQwH|Wgv5JY-AuPZ*FBEVRRs2AarSMWho$cXlZO^AaHVNV`U&dJs@UmZ*Ov6"
    "aB^v5WiBEjA_^cNARulaJs>a&ARr(hcW7yBWgu=KJRoghcwcpHX>@Zs3LqdLARr(hAaHUZJs@pia%*LDUvP41V`V65bY*QUAZc!P"
    "DGDGUARr(hARuXGAaHUZJRoLlZ*Ov6aB^v5WjYEVARr(hARr(hARr)Oa%Ev_3LqdLARr(hAZ{QlJs>d(ARr(hARr(hX=WgBav(k+"
    "F**t$ARr(hARr(hARr)VZgwCmJs>d(ARr(ha%FUNa&91Q3JMBjWo95|X<}bvZ*z1gZYeqnARr(hVJskGAUz;4EFduoARr(hW^ZyJ"
    "Um$62AaY@DXJsgEDLM)uARr(hARr)NEFfYaJs@H%AYmXYAYuw2ARr)eWps6NZXjU_3JPRpW*}&3a%Eq1Z**a7C~he_3LqdLAaZ4N"
    "b#iVXb9HSfW@%zyV{dbGC}}AmW^ZyJX&`BCAaY@DXJsgEDJcpH3L_vbEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElA"
    "EiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiEkyBOqsGZ*66Ca(N0PAT2E|EiElAEiElAEiElAEiElAEiElAEiElA"
    "EiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEed31W*}r~b95+SEFfYjItm~lARuyObairWAYo#2"
    "C}CSLT_7zWVp}j>DIhB#VPbPAVOud>AT1zbTQOZJ3JMBjWo965Wnpq<b97&GXk}z5a49+pARr(hVr6r5EFfZJAUz;cNJT_nR7p%l"
    "Q(G`yEFd{K3LqdLAZBlJAafvTZXi=gMMPgzNlZmkItm~lARr(hARuHQJs@Ohb95+hEFg0!3LqdLARr(hAZcbGWFR~sVq`iBARr(h"
    "ARr(hARr(hVr6r5EFfZJAUz;+EFfeGARr(ha%FUNa&91EWpi{aAYx<+3JPRpW*}o_ZggdGUu0=>bSQKwItm~lARuyObairWAYo#2"
    "D0EveT_7zWLq$$hMN(TZT`3?dAYo#2D0EvfT_7zWLq$$hMN(TaT`39*3S?zwAZ2x9Y-waND0C@03LqdLAaZ4Nb#iVXD0EveT_7zW"
    "Lq$$hMN(TZT`3?cDj+f-D<CLzTQOZAEg(ZhPE<uwTQOZJASx;#G71U`WMyU`b97~JUvzJGVRB?Ba4aBkDLM)uARr)Qcq|}fc_2L?"
    "aa%B5AT1zpTQFTLAaPqUT_7zWa9c553LqdLAZcbGWOyJwJs>b3VQyp~WO*PxJs>bT3LqdLARr(hAaZ4Nb#iVXPH%2y3LqdLAZcbG"
    "VPbPAWOykcK0P2|Vsj{Dc_}&yARr(hARr(ha%FUNa&90ZML|<kA|PpIAY^zTJ|Hk4Wo&b0AR<>qQ&b`fARr(ha%FUNa&90ZQ%_Y?"
    "NFpF<W*}sFAU+^4AZ2WGWgsF>Pf}D!A_@u$WMyU`adly2a$#<CD0nO&c_}&yARr(ha%FUNa&90fB2FS8X=WgKAUq&7AZ2WGWgsF`"
    "A}Js%ASfbNA|PpIAb21=AT=OmY;$EGB1Ix83JMA%AT2E|EiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElA"
    "EiElAEiElAEiElAEiElAEiElAEiElAEiElAEeazbaAk6HX>)XCZge1UWpXWWY+-q2av*bbVRU5*BOom;EiElAEiElAEiElAEiElA"
    "EiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiDQ`PDxEcOkY}0PDLO+ATcruK~70c"
    "K}=spK~i5#Nli&qAUz;oYi@61C?ZQoUqMbuO+idwMnO^{EFd&4FewUCNJT|ZUqeAqAUz;oYi@61C?ZQoUsFg$MNnTuK~N$rATcm0"
    "3R6f$MNnT*MN(f`K~hd2Js@9eZf{~JB1=bKQ%FTcP+w3*QeRm?QcfZ)ATucnQ%FTcP+v?zQ&e9<RaqcCAYW^4Z(=ASOGjT*NJT|Z"
    "Ura$$R9`|>St2YTGBGI%O+`#kPBLFjL0BL?AYW^4Z(=ASOGjT#MNCglGG9$WSRyPSF)%eL3Qa{!PfjvlMpaBqAUz;oYi@61C?ZQo"
    "Urj|!PfjvlMpaBqA}k;`H7N>HR8m1#UsFXxR8LYMJs@9eZf{~JB1=bKQ&dtxS6@>_LsU;vA}k;<DGF0mQbAW=Q$<5mPf}ALJs@9e"
    "Zf{~JB1=bKQ&dtxS6@>_LsU;vQz9%NHYo~2K~hprR9{0uP#`@ZUu$k}Vkja@M_)rhQc_P;UqeAqA}k;=GARl}K~hprR9{U=PG3Vo"
    "MnzK~Js@9eZf{~JB1=bKLqSqfPgGw`NlsrwK}JPWA}k;>DGEbDQc_P;Ura$$R9{d`K~7X4Js@9eZf{~JB1=bKLqSqfPgGw_K~q#B"
    "EFdy9DGF0mQbAW=LqSj=Js@9eZf{~JB1=bKQ&dtxS6@RxP$DcKG%_g)Q&dtxS6@s)Q&eA2OhHamAUz;oYi@61C?ZQoUsF_4L04Z)"
    "K~q#BEFdv6DGE<eMNVH$MNCglQy@JcF)<2HP(@B(Q%FTcP#`@ZG73*nMNVHsPgheQJs>g)Pf$fpUsp&)K~!H;MMXqYAUz;93Qtf)"
    "PG46@ML|?wMny$LAUz;73Qtf)PG3k#QbkiBJs>p-3JPOvVRLgJQ*>c;WjYEVARr)QWo95>UukY>bYEX6b7gF1DLM)uARr(hARr)f"
    "Wo%|HY+-YBUvqS2a3DP(EinopARr(hARr)fWo%|Ha&K&9b09q+dwn1vARr(hARr(hARr?kD0nP+DIhIAAR<9dNligaA|QMqB27h1"
    "Pfj8rd>|rIR8m1#A|QMqB3DR7K~y3NARr(hARr(hb7gF1E_7jXXJvGAAUz;^eIOtpARr(hARr(kAa!nObRcPDcpxo4ASie&c_|=t"
    "VRC0>bRcYDb95kdb#iVBARr(hARr(hb7gF1E@5tCXm4_JAUz;^eIOtpARr(hARr(kAa!nObRcPDcpxo4ASie&c_|=tWpZ+9bZ>Hb"
    "AYpD}Xm4^LW^ZyJbZ=x~c?uvPARr(hARu#PY-TQPWo&P5Uvz0~Wpf}sAX{AuARr(hARr(hb7gF1E^TFOZ*E_3Y+-J6Uu0o<AUz;3"
    "ARr(kAY@^AAa8CUcW7y2XdrE6Y;SHLbZKm5b0BSDc_3nCAaHD9Zggd23LqdLARr(hAaiAGW-e`IY;SH~cVTvAG9W!5MqzAoWeOl5"
    "ARr(hARu#PY-TQPWo&P5Uw2`4WinrGAUz;4G72CdARr(hARu#PY-TQSY+-I+WMO$AJs>SH3LqdLARr(hAaiAGW-e%Ha%FR0cVTXH"
    "Wn>^dATSCbARr(hARr)fWo%|Hb98cHcVBd2a%W|9AUz;33LqdLARr(hAaiAGW-eo4a&m8UUvyz|XJvFCJs>a&ARr(hARr(hb7gF1"
    "E@N+ZUvyz|XJvFCJs?j|MNVHsPghe4ARr(hARr(hb7gF1E^}yQWpH0~VRC0>bRaz-Pf$fpUsFg$MNkSLARr(hARr)fWo%|HVsCY4"
    "Xmnq6Z)9P4AUz;^eF`8TARr(hARu#PY-TQSWo~3?Zf9R%ZfR{{Y;zzzAbWivAR{1QZfR{{Y#=Q@AY*TJZge1GZ*^yAbRc4NbRceT"
    "bRc<UbRckSVPj=vAShvQaB^>WDGDGUARr(hARu#PY-TQOZ)YGqAX{Au3JPCSR6$flQy@JcdwmKD3S?zwAZKNCUvqR}bY&=TY+-q2"
    "ax5TobY*ZUItm~lARu#eAUz;oQ&d4zMN=+kWppTTY+-q2aw!TRARr)VW*~EPAZc?TPH%2yAa8OYb97~JAUr)FFd%PoAaitOa3DM&"
    "b963jVRLj}b97~JItm~lARr(hARu#eAUz;cbYXO5C@BgcARr(hARr)LQ&d4zMN?aFY+-q2a$O)jAaisIARr(hb963jVRLj}b97~J"
    "AUz;+bY*Y~ARr(ha%FUNa&91VbP5UzBOom;EiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElA"
    "EiElAEiElAEiElAEiElAEiElAEiDQoAa7!GWpZ|5bZKvHAZTT5aAk6H3L_vbEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElA"
    "EiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiEkyWMyU`M<`@0AZsijWMyVyb!>D!PH%2yDLM)uARr)VW*}rB"
    "X>%Y>Z*FBe3LqdLARr(hAaZ4Nb#iVXWMyVyb!>DBARr(hbaHt*3LqdLARr(hAa)=<AY?9QWppTOEFffMW?^+~bSWTdW*}%`b76FJ"
    "awudhAR=dFbRsDrWo&b0AZKNCVRUqIC}b=kYb+pSWoBV@Y;-9KARr(hWq4y{aC9I=cw=R7bZKvHItm~lARr(hARu-iJs@OdW?^+~"
    "bP6CKARuyObairWAY^4`VRdYDAZcbGb|7hUAWm;?Wgum2b7dfQ3JMBjWo960b6;?5VQzFNbSXLtARr(ha%FUNa&91Lb7^jKbYX5|"
    "WhitkAY^G{bSWTVZe$>IE@x$QC?acVZe$`UAU!=GB2Y|0PE;Za3JPRpW*}*EUw36?WGHkgItm~lARuyObairWAZc@HZgX^DZewLA"
    "bSxlbX=8LLAYpD~AapKgWppSaYiVv|A}Js}Js=`iMMXp+3JMBjWo960b6<0Ga&=>LD0C@03LqdLAaZ4Nb#iVXX>)0Ab97;DV`V6G"
    "EFffQV{|DXVQyp~bS`IQbSNTgX>Md9DIjTXASfb3Pft)HEFdCKK~q#!Qbi&u3JMBjWo95~VRK($ZfR{{Y$$XoItm~lARuyObairW"
    "AZc@HZgX^DZewLAbSxlbX=8LLAYpD~AR=LIX>DO_A|PpQAao#MZe$>IE@x$QC?a8QX>DO_A}Jtgb0BVSbRbS|Ze<Dz3S?zwAZc@7"
    "Wo>YDd0%sMa&=>LD0C@03LqdLAaZ4Nb#iVXX>(t5baHiLbSQKwAYpD~AZ~ATAZTH8Utw-(ZDDLEbSVl73S(?xb8{e9Z*pv8Itm~l"
    "ARusIb8`v`3S?zwAaG%Fb7d%RVsj}v3LqdLAXgwgAXjg4Y-A`Y3LqdLAXhGMY+-q2av(h*X>N2VM<{P%b1WbtaBN|DWpW}cATTK@"
    "3LqdLAXhGPbY*ZLJs@drbSOtCZ(?&SAR=>gWpE-aATTK@3LqdLAXhGAVR;}uAZc!NC`Tx7Vsk7YB4lBCA}k<RE^~Bca3C))AXHF9"
    "DJcpdARr)DE@*Fcav(h*X>N2VM<{P%b1WbtXm53LA}k<RE^~Bca3CchR8T}IDGDGUARuO8a&2=UJs?LYZ(?&SAR=aAa&2=WEFfE5"
    "DGDGUARt#RW?^z|AUz;vVRCJATURb{Y+-q2a$O1_ARr)DE^lyfAUz;vVRCJATQMLlAXhGMY+-q2a$O*4W*}^3ZYXA9a&2=dAU+^5"
    "AZ2WGWgt#(Ze<D}ARr)DE^TjaWqBYyAZBcDVRR@*C|53KVRCIOAR=vVZe@8QEFdr`DGDGUARt#RbZKm5b09q+M<`b=W?^z|EFdCu"
    "X>4V4A}k<VT`3A6ARr)DE@okJZDn#GJs@;-aBO8LM<`b=W?^z|EFdCgVRCI{aw04sC^Re}G$|=53LqdLAXhGEVQyq|AUz;kbailS"
    "WhiJVAZBlJAZQ?IZXib}S1x8@a&0UiB4}Z5WOE`cAX{B2T?!x|ARt#Rb#7^Nb09q+TURb-VRCI{a$O)RAXhGEVQyq|3LqdLAXhGR"
    "ZftL3Yh`30Js@mpb95+2C|53KVRCIOAR={cY;R+0Wn^D*bzx+3VQzGDA}k<VB2HH#T`4IFARr(hS1xF2a%FR0bZ=x~c_2L?X>N2V"
    "M<`b=W?^z|EFdCiX>w(AUvzI|VR<4fATTK@3LqdLAaHVNb|5_<M<{P%b1WbtaB^vOVRU68EFgP*DGDGUARt#Rb7*B`AUz;tX=8LK"
    "M<{S|X?83iB6DbEWFjmedwnS>3LqdLAXhGPWo2Y@AUz;tX=8LKM<{S|X?83iB6DSBWOE`cAbWi&DGDGUARuXOc5@&-AV(;0a%pxf"
    "AR=jQc4cmKZ*pm6b0RDtTYG(7DGDGUARt#RX>N9NAUz;kWNBk`C~0;nAZBlJAZd0WX>K5CZgz8B3LqdLAa`hKY-J#9Wo{@}E@^Id"
    "b15J^AZ%rBC|53ZZfSILDLM)uARr(hARr)DE@^Idb1q?UaAj^}D0_V=3LqdLAZ=lCYh`pGJs?LYZ(?&SAR=vHa%*LDA}k<#eJKhc"
    "ARr)DE@^IdAUz;tX=8LKM<{J!a%*LDEFdCjZgypEbZ>HbA}k<#eJLplARr(hS1xdJX=7z`AUz;tX=8LKM<{J!a%*LDEFdCqa%p2_"
    "b0RDtdwnS>3LqdLAarkcZXi7%M<{P%b1WbtbZ>WVA}k<#eJKhcARr)DE^}yaaC0C%AZ%%KbSOtCbZ>WVEFdCvZftL3Yh`3#b7*gH"
    "b0RDtTU{wB3LqdLAXhGPbY*aJUu<P&bRaz-OhHprUsF^?P#`TJS1xmOWpE%MARr(hARr(hBOqa8bZKvHb0A@6bY*fNbZBXFAa8DE"
    "AarP9bRctdX>4pDWq4&{b#!G4ARr(ha%FUNa&9133JMBjWo96BX>4U*VRR^0EFf?xItm~lARu@wAbB7?AaDvGARr)VW*{&iJUt+I"
    "AUq&KPeD>dAYpD~ATS_2Js^1?JRm|(K~h9I3LqdLARr(hAaZ4Nb#iVXS1xpEY-Mv>d0kt0T?!x|ARuyObairWAR<gpLrX<OA_@u$"
    "WMyU`b7*gHUt@1|ZgePDEFf@lZ)A02bSXLtARr(hZXi7%FbW_bARuOMav*acX>K4_E^}yaaC15eARr(hARr(haB^>Cb09q+Q%Fxx"
    "Q!Zy^bSQHyAX{B23LqdLARr(hAZcbGaB^>Cbz^iOX>K5Ja&KgFItm~lARr(hARr(hARulaD?K1GAZcbGY-MgJaB^>Cb15J_Js>e4"
    "Wo&b0ATbIcARr)eWps6NZXj+73JPRpW*~EDWn^D;Z**a7C|4;u3LqdLAaZ4Nb#iVXb9HSfb|7YNav*jfX>K4_E^}yQWG;4LY;|RG"
    "C@COmW*~Mc3JMA%AT2E|EiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElA"
    "EiElAEiElAEeazbbZKm5AaZYPWpf}eAZ%fIZ*_DEBOom;EiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElA"
    "EiElAEiElAEiElAEiElAEiElAEiElAEiElAEiDRUWo96CZftL3Yh`3#bZKm5b0}9SItm~lARupbbRaz-TU`nuARr)SZ*m}cAZczO"
    "a$#;~Whg>VK~h91Itm~lARr(hARuOMav*phX>K5LVQyz-C_+y`QbZ{_3LqdLARr(hARr(hAZcbGS1xpEY-Mv>d0kt0T_7PnAR<gp"
    "LrX<OB035nARr(hARr(hARr(hARr)bb#yLaaByXAWGE<jEFgI)DGDGUARuyObairWAa8Ya3JMBjWo95@b8~5DZeMb5Y-MvOb95{q"
    "S1CFQARr(hA|fIvQe`P)Z*6dObY&oNZ)|0AAZBlJAYp85Aa!nRZ)0m^WFT~DY-MvUAVF?vZDDL6dT(xJAUz;%Wnpq<b95kdX>4V4"
    "AarjEARr(hV`Xl1WpXSabZBL6AaitbVRvF>a&mbfa%pa7EFg4fWo{sMXk}q^E+9>1Y;SHLbZKm5b0A@IWgup0cx7ZDZ*CxDVR;}h"
    "E+Qf#3LqdLAarSLWpf}sAa!nRZ)0m^WM6b?Y-MvOS1Af0ARr)aUtw-(ZDDL6Js@picqnspE@N+ZUvyz|XJvFCD<E@pE^}yQWpH0~"
    "VRC0>bSxk=DGDGUARuXGAXhGAVR;}vJs>tZ3LqdLARr(hAZ}k_ZfR{{Y#==#ZDDvQZeL+;X>DO_EFeKnNligaUs_L2MJWm(ARr)R"
    "Y-wg7S1x2>c_2PLATl}%ARr(hARr(hZeL+;X>DO_AUz;$VR$HRUtw-(ZDDLIAUG)sARr(hBOqmXX>)XGZf782ZfR{{Y#?)Va&=>L"
    "b#i5MAYp8GVR>^PYh`6{AarPDAVE$^O+ic`a&K&93LqdLAa8PHWpZU?AUz;+Z*p{HWGHlLY-MvSAZulLJ#1lZVq{?;bUGj?V`Xl1"
    "WpZC+X>)WabSW$#Wp!h0X=E}ebSW$#bXzf9EFg4SFkLAr3LqdLAaZYPWpf}sAbWiZARr(hX=Wf-E@WYOAU!=GFd$)WWFT&DbRctd"
    "E^TFOZ*E_7X>4V4Itm~lARr(hARr?kZDnk4ZgU`SZXk4MWguo@a&%~Ab95k1S0Hq0Y-MvGC~|Lib09D+G9Yblb98KZDGDGUARr(h"
    "ARumcAUz;kbRcGLav*ddX>K5Ia%5$4Wn>^}W*~8OVPtY)ZgeOrbSWS`Js=`ZS0Y^sARr(hARr(hW?^z5Js@*$a&%>6C~kKwAZulL"
    "J#1lZVq{?;bUGj?En{VFbY*g1WNCABD0C?-Aaq+XT`VAUTQFTIDGDGUARr(hARu#eE^TFOZ*E_7X>4V4AUz;vVRBnKPf$fpUrj|!"
    "Pfk-^3LqdLAZcbGb963kWo&P5Uw2`4WilXPZe$>CZ*(AYbS`aWY;SH~bZKm5b2<tjARr(hARr(kAaiA7Z*F8DcVTvAIv{3Ya&%~A"
    "b95kPa%E*8bZKm5b0BYKAarPDAa!nRZ)0m^WFTf?a%~DAARr(hARr)SVR9fnAaieWbY)~HTXZ01Z*m}XAZczObZKm5b0BGEAarSL"
    "WnW=*C|4{XbSWTdb0AJ{Ze<{Eav*7QUw36?WGHlLY-L|zbSPIWAap4yT`US9ARr(hARr(hARr(hARr(hARr(hYh`&oY+-F;WMLq5"
    "Iv^-5V`Xl1WpZC+X>)WabSW$#bXzf9EFg4SFkLAr3LqdLARr(hAaissZDnk4ZeMh1Y-MvGJs@Uba$7ocbS`aWY;SH~cVTvAGGA_8"
    "3LqdLAZ=xAZ*E_6Wpp4tAaiAOD06f!ZDnk4ZeMh1Y-MvP3LqdLAYpE4ZDDL*Y-MJ2AUz;%Utw-(ZDDK*ARr(hW^ZyJbRcPNAa8PH"
    "WpZU?Itm~lARr(hARu&XAUz;-X>4U*VRR^0EFg3#3LqdLARr(hAZcbGX>(t5baHiLbSQLeDLM)uARr(hARr(hARr)eZ)|0ATXbC@"
    "Js=`MPDxEcOd<*(ARr(hARr(hARr)NZfR{{Y+r0;W^^DeJs>d(ARr(hW^ZyJbRcPNAa8PHWpZU?Itm~lARr(hARuXGAao#UZXj}R"
    "Y-Mvg3LqdLARr(hARr(hAY*TCbZKsNWeOl5ARr(hARuXGAao#UZXj)CY;SH~b7gco3LqdLARr(hARr(hAaraXJs@;xY-L|zbSPIW"
    "Aap4TARr(hARr(hARr(hX=WgEbS`aWY;SH~aBN|2bYEm)c_2MKAXhGAVR;~Lav&&ab6;?5VQzFNbZjXgVQyp~bZjnXWppSaV{&hB"
    "A}Js}Js=`YMNCglA}KlwARr(hARr(hARr(hARr(ha&K&9b6a#>AUz-=O+`#kP9h2*ARr(hARr(hARr(hARr)PZ*FvHZgph}ARr(h"
    "BOqaJX>DO_AbM|ZWjY{pbaH89bZmJbbZBKDZe?L|Wpi{ObZKm5b0BnYAarPDAaiJCWFRPVWoKb>WNc+~b0BYKAa`hCbP6CKARr?k"
    "V|8+JWo~qAc_3$UZ+CMbbZBLAWgsmrAY*cGaC0DZXk~I`AYpQ4AZ~ATAaZ4JY+-J6Wn>^>Ze$>IXk{RDX>4U6X>$r7ARr(kAY*TC"
    "c4cyOWn>_9Zy;eHaA9+Fb#i4OZ*F5{AZBu9WhpKoAWLOsaC0DKc4cyTAY@@_Y<VDIZfR{{Y#?@Nb7^!Sb7*gJbS?@YARr)QWo95|"
    "a%E*GbSXLtARr(hARr(hbZj6!AarSLWnW=*C|4{XbSVlTARr(hARr)eWps6NZXk4QAZc?TPH%2yAa8OYX>(t9Wo2Y2bZjXKARr(h"
    "V_|M&AUz;kbRcGLav*ddX>K5Ia%5$4Wn>^}W*~GRZf|rTX>K5LZ)|0AT?!x|ARumEdT(xJAUz;+b!{jyAZBlJAaWpSZXj}RY-MvU"
    "c42IFWpgMgAZcbGav(iDAR<9dNligaA}I<WARr)SZ*m}XAZczOV_|M&Itm~lARr(hARuXGAYpE4ZDDL*Y-MJ2AUr)FFggk#ARr(h"
    "ARr(hARr)Oa%Ev_3LqdLARr(hAaraXJs@;xY-L|zbSPIWAap4TARr(hARr(hBOq>Nc4cxPV{Bn*Z6IMFXL4_MX>MmAb98cHcVcC7"
    "a(N&qF*hw_VR;~9a&K@cAZBlJAarPDAbM|ZWeOl5ARr(hARuXGAZc@7aBN|2bSQLeDIj5PWFT~GE@x$QC?aEWZ*U?hAU!=GB2!dS"
    "L03XWQc_tWItm~lARr(hARr(hARuFJZggpGb!7@5ARr(hARr(kAYx^CZ*F8DbZBKDV{dY0AbM|ZWgu^EY<VDIV`F7-bRcwTY-MvG"
    "bZB98AYpQ4AaitSY-}K7Y;SXAAarjabZBJ?ARr(hARr(hBOr5VWn>^NEg)eaW?^z5aA9+Fb#i4OZDnC@b0A?LY;SI7AY@@_Y<VDN"
    "Wo2X_baH8MAYpD~AZ=-Lb7f>8W@Tk$a|$3JARr(hARuXGAZ}lJZ*FBEK0P2oPDxEcOkY}0PDLPLZe$>1Wo~q3a$jU=b95+lDIh)|"
    "K~70cK}=spK~i5#Nli&qItm~lARr(hARr(hARuFJZggpGb!7@5ARr(hARr)eZ)|0ATXbC@Js=`MPDxEcOd<*(ARr(hARr)NZfR{{"
    "Y+r0;W^^DeJs>d(ARr(hARr(hZeMzDZe<`VJs>d(ARr(hBOr5ha$$F3WpZ+9Wpg?pWq4_GbZKs9AaHD9Zgg`XW@&PBbSxlrXk~67"
    "W^!d^AarSLWpf~IWnpq<b95kdXk{Q{Wo~qGWeOl5ARu#ea$$F0Y-MJ2AUz;+bS`sra$$F0bYXI5WpoN4ARr)SZ*m}XAZczOZ*pX1"
    "a%E&X3LqdLARr(hAZcbGbRceTbRcPNAaZYPWpg?TARr(hARr(hARr(hbZj6!AarSLWnW=*C|4{XbSVlTARr(hARr(hARr)VW*}*E"
    "UvO+;ZgePgY$+gNZe$>IY%XVIbSNTYa&K@VDIh&PAR<##QbAWjMN(2(B035nARr(hARr(hARr(hARr)eZ)|0ATXbC@Js=`eR8m1#"
    "A_^cNARr(hARr(hARr(hARu#ea$$F0Y-MJ2AT2#0F$y3cARuyOb95j*AX{`GW^ZyJbRcPNAa8PHWpZU?AZcbGbRceTbRcPNAaZYP"
    "WpiB$ARr(hX=Wf(R8m1#UsFXxR8LYm3LqdLARr(hAR{1VX>4pDb98cHcVcC7a%p9AAaiA7bZ>GXVtF8QWn*-2av&&SZfA9DVR9gM"
    "Wn^b%b0A@IZ*^{DAarPDAaiJCWGNtXZweqFARr(hARr?kbZB98AaHD9Zgg`XZ)PBLXk{RCVQpm~VP|C^EiE8-XlY|;AZ}%4WFU88"
    "bY*fNZ*CxTXk{RCVQpm~WMO%8AT2ElARr(hARr(hBOr5WbRceJcyu6iZy;r1V`w06bZBLAAYpD~Aa8DEAZTH3WFTW<ZXk1aWo2+6"
    "bZBL5E(#zZARr(hARuIAW*~EGWqBxcDLM)uARr(hARr(hARr)NZf77pAZ=lEXf9!NVQw-gbXzf9AT1z6MNU*jQd==yEFg4SFkK)m"
    "AVWn?R7Fx-FkLAMARr(hARr(hARr(hb7f;7Js@drbSNlcZf77XAZ=lEXfAMRDIhN(C^8@_AZ=lEXfAMRDIh8!Q&dtxS6@>_LsU;v"
    "Qz;-NAX8LQL04Z>MMG3iQd0^bARr(hARr(hARr)eWps6NZXhTyAZcbGW^!d^D0C?xWo&b0ATcZ;b7f;JAY)~2bY*g1WNCABD0C?)"
    "3LqdLAZ2WGWjYEVARr(hARr)QWo96AYh`&TbSXLtARr(hARr(hARr(ha%FUNa&90fFd%7WAZBu9WhithAZ2WGWgsyuAY)~2bY*g1"
    "WNCABD0C?)3LqdLAZBlJAao#UZXk1Sa&%>6C~{?UbSxliWqCbwYh`&UItm~lARr(hARuXGAaitbVRv6_WoC3BJUt*VItm~lARr(h"
    "ARr(hARuCLWnpUyARr(hARr(ha&K&9b6a#>AUz-=Q&dtxS0V}^ARr(hARr)fbaG*LUu<P&bRaD~ATbIcARr(kAY);2a&L5VASiEc"
    "Y<VDeXk~67aAkBLV_{}xb0A}KWnpw>AY^52VQyq8Iv{0uX>)XGZf784VRCYBbRckSVQzGDAZul1a0(zGARr?kbZBKDa&K&9EFg4f"
    "Wo{s5a%E*8bZKm5b0BVIVRB`2bRcwSWguf^Zgg^G3LqdLAY);2a&L5BY-MJ2AUz;+bS`6Ia&m8UUvyz|XJvE>ARr(hW^ZyJbRcPN"
    "Aa8PHWpZU?Itm~lARr(hARuXGAao#ZZ*(AOZXj}RY-Mvg3LqdLARr(hARr(hAaraXJs@;xY-L|zbSPIWAap4TARr(hARr(hARr(h"
    "X=Wg4b6;?5VQzFNbZjXgVQyp~bZjnXWppSaV{&hBA}Js}Js=`OK~hprR3bVGARr(hARr(hARr(hARr(ha&K&9b6a#>AUz-=LqSqf"
    "PgEibARr(hARr(hARr(hARr(hV_|Y~Z**U5WoC3BEj=JH3LqdLAaZ4MbRaz-TXZ01Z*m}XAZczOZ*pX1a%E&7X=WgFAZ~ATAZczO"
    "a&K&9b6pA`ARr)SZ*m}XAZczOb8m8VWn?IFWpi{aAZulLJ#1lZVq{?;bUGj?Fd%7WAZBu9WhithAZ2WGWgsyuAY)~2bY*g1WNCAB"
    "D0C?)DLM)uARr(hARr)VW*}o>a&m8UUu<P&bRaxEATT-#ARr(hARr(hARr(hVsd3+YYHGBARr(hARuyYY-Mv>bX_1lAR<FSQc_P;"
    "A_^cNARr(hARuF5a&m8UUu<P&bRaD~ATbIcARr)SZ*m}XAZczOZ*pX1a%E&X3LqdLARr(hAZcbGbRceTbRcPNAaZYPWpg?TARr(h"
    "ARr(hARr(ha&K&9b6a#>AUz-=S4c%cR3Zu>ARr)fbS`plY-MvGJs@&#Y-Mu_3JN12EiElAEiElAEiElAEiElAEiElAEiElAEiElA"
    "EiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElA3L_w7VQFl6AZ=k|a&I7TY+-H+BOom;EiElAEiElA"
    "EiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiDRUWo95_Z*^{TUtw-("
    "ZDDM4C|4{XYiVv|Jx*_KWh@|OVRCIfPH%2yDLM)uARr)SVRCICJs@Uba%~`KW*}x^a%~`Kb0BVSbRbS|Ze<{4Y;$EGS1x8@a%~DA"
    "ARr)gX>4V4AUz;QC}v@DZ7d)nbZKm5b0RDtTU{v%ARr(hZXi7%FbW_bARuOMav*YVcOYqQAarSLWpg?TARr(hARr(hW^ZyJbRcPN"
    "AaZYaItm~lARr(hARr(hARuXGAZTH8Utw-(ZDDLEbSWTVZe$=RYiVv|AZc?TPH%2yAa8OYbS`IQbSNTWZfR{{Y$7QjJv|_6X>MdG"
    "Itm~lARr(hARr(hARr(hARulaD?K1F3LqdLAaZ4Nb#iVXZVCztWMyU`V{dhCbYEj~Z*VAAEFfcYZ*VLiW?^z|Jx*_KWhpueARr(h"
    "W?^z|AUz;vVRCICX=Wg1VRCICX>%ZMZ*(9|Z*FBEWo&b0AXhGCVRCH>ARr(hZXi7%FbW_bARuOMav*YVcOYqQAV(->VRCIOAR=^W"
    "Y-MvIEFfE5DLM)uARr(hARr)SZ*m}XAZczOa&LD!3LqdLARr(hARr(hAZcbGX>(t2Y+-J6D0C?xVQyp~bS`IQbSNTYa&K@VDIh&P"
    "AY*cGa5@SgARr(hARr(hARr(hARr)aAS*o}F$y3cARuyObairWAZ`i@3S?zwAa!tLVRU6*bYXI5Wpr~Wb95{qS1CFQARr(hWMO$A"
    "Js?*uWMO#<ARr(hd0}#HAUz;+Xm4;|V{dhCbSPIWAR<>!PfQ{yATKW<G9VxzARr(hARr(kAbDYOZXk1XZ*pZIV{dhCbaNnNZ*^j9"
    "WeOl5ARujNY-?Y0Xm4<HAUz;+Xm4;|V{dhCbSPIWAR<jkOiLmu3LqdLAaitbVRv71Xm4<HAUz;+Xm4;|V{dhCbSPIWAR<##QbAWj"
    "MN(2(A}I<WARr(kAY*TLa|$3JARuXGAY@^AAUq&6Itm~lARr(hARu#eE@N+ZUvyz|XJvFCJs?j|MNVHsPghe4ARr(hWo&6?AY@^A"
    "AUq&7Itm~lARr(hARu#eE@N+ZUvyz|XJvFCJs>g)ARr(hWo&6?AY@^AAU!=GH986)ARr(hARr)fbS`6WcVBd2a%W|9AUz;63LqdL"
    "AZ2W6W*}r?c_2MKAT~M*ARr(hARr(hb963aZ+BmGVRC0>bRaz-H3}dgARuLIX=Wf~VR;}uJs>wa3LqdLARr(hAaissV{dm~bYXI5"
    "Wpp4tAU6siARr)RY;$Eg3LqdLARr(hAaissV{dm~bYXI5Wpp4tAUGgtW*}{8Y-?Y0Xm4<HAU-`HF(74Zb7del3LqdLARr(hAZcbG"
    "WMO$AK0P2YG9Y1YWFT#6Y-?Y0Xm4<HAU-`HG&%|(ARr(hARr(hARr)fbS`6WcVBd2a%W|9AUz;C3LqdLAZcbGWMO$AJ|HnQItm~l"
    "ARr(hARu#eE@N+ZUvyz|XJvFCJs@pqZYXnfE@N+ZUvyz|XJvFOAa8eWWn^DrZfR{{Y;!1AEFdC7Pgf!-DGDGUARr?kb7*B{a5^Am"
    "VPj|@d0}#HAaitYa%CW8VRUmKF)|=`Z*OcbWMO$AD1JF0b7*B{a3FVYa&%}ZJ0NpoVQgg}cWHEJAarPDZ3-YDARuXGAY@^AAUq&8"
    "Itm~lARr(hARu#eE^}yQWpH0~VRC0>bRaz-Pf$fpUsFg$MNkSLARr)RY;$Eg3LqdLARr(hAaissb7*B{a9?y`a%W|9AUz;$VR$G{"
    "P(@B(Q%FTcP%I#AX>KS}NJT|ZUqeAqEFe!%MNVH+NJT|ZAS)nKNJT|ZUr<F-Us*v?P9Q2Ed0}#HDJcpdARr)VW*}r?c_2O@Q%FTc"
    "P+v?zQ&e9<RarU;ARr(hARr(hb963qXk}$^Uvyz|XJvFCJs@pqZYXnfE^}yQWpH0~VRC0>bSxlmcWz~5Utw-(ZDDM4C|4{XB2!33"
    "MNlFsDGDGUARr?kb7f<1Ze$>BWo&P5Aa`MQWjY{lZftoVX=WgFXk{R6Wo&P5AZ=lCYh`pGX>%ZMZ*(AMY;|;WWn>^>W^`q8AY@^A"
    "ATclsARr(hX=Wf~VR;~FZXhTzFf1T3F)1KnZe$>CZ*(AYbS`aWY;SH~cVTvAG9Y1YWFTX2b#8QDV{&hBC|4{XB27h1Pfj8!AU!=G"
    "Fggk#ARr(hARr)ZWo&P5UukZ3AUz;gE@^IdE@x$QC?ZWoOixZCEFeumQcFctUr8`23LqdLARr(hAZcbGZDnk4ZeM9`b|5@GAWcD1"
    "OGQ*)NiZNQAWcO~PfjvlO+i>7VQyp~S1xUDZe@8OK0P2ZFfcGW3LqdLARr(hARr(hAaissZDnk4ZeMp{c4aakJs?zab!7@5ARr(h"
    "ARr(hARr)fbS`aWY;SH~cVTvAGGA^WJs>eMAZcbGZDnk4ZeM9`b|5@GAWcD1OGQ*)NiZNQAWcO~PfjvlMpaBqAZ2WGWgs>RARr(h"
    "ARr(hARr(hb963kWo&P5Uvz0~Wpf}sAX{AuARr(hARr(hARr(hb963kWo&P5UvO+;ZggK{VR;}uAY@^A3LqdLAR{1ibaG*LVr6o2"
    "X=QUdAaHD9Zge1JVRCGFASiTbWqBZTWo&F9VRRsKV_|Y*X>@rYaB^v5Wpf~Aa&K)QWMO$Aelas3Z*D0HARr(hBOqaJWFTj9Z+9Sc"
    "Xk{Q~X=Q9=AYpSLY+-I>ATJ<eVRL98VRCY5c4aOgAX8{>aC0DSZ)A0BVRU66bZBKDW@&C=Y#?)KdSxyOARr(hX=Wf~VR;}tATv4&"
    "ARr(hARr(hbZ2xRJs>a&ARr(hWo&6?AY@^AAUr)FH986)ARr(hARr)gXLKMvAT|mhARr)RY-wg7WMO$AJUt*cItm~lARr(hARu&S"
    "bRaz-F*ph!ARr)RY-wg7WMO$AJUt*eItm~lARr(hARu&SbRaz-GcXDuARr)RY;$Eg3LqdLARr(hAarMRAUz;6HVPmhARuXGAY@^A"
    "AU-`HIXVg;ARr(hARr)gXLKMdJs>k6Dj;)oa$$F0b7*gHb094sC^R5xW*~EPa$$F0b7*gHb09rEATS_hY;$EGFewTkARr(hARr)g"
    "XLKMvAZ=lIC^9rGAZ=-GC{t8YL04ZxK~O9pbZ2xaDGDGUARumEZf|!WJs@Lmb#8QDV{&hBC|4{XB2!dSL03XWQc_tWDGDGUARuXG"
    "AY@^AAU+^dR8m1#Ura$$R9{d`K~7XU3LqdLARr(hAarMRAUz;%Uv6)A3LqdLAaissb98cHcVBd2a%W|9AUz;$VR$HXXLKweZeMP1"
    "cPR=WARr(kAY);2a&L5VIv{XmbRc74W@U3AWnpw6F)}Y?VR;~BVPj|@C}?SJXJsIBV_|Y*X>@rYV|8+NWgsm+AaHVNV`U&?Y-w#`"
    "a|$3JARr?kb97~8aBO)XZ*F5{AarPDAarkcZXjr3b0B1LVQFq<WFT#Ca%CWNXkl(3G&L|OJ0NCZa&2XDb0BSDa%*LDb09V^WMO$O"
    "3LqdLAaG@LUt?irWpf}sAaiwXC@~;rZ*m}WAZczOS1xmCZ*X%UX=WgEAU!=GB2YzCUqeAgMItE*ARr(hX=Wf~VR;}tJs?9tQc_P;"
    "Ura$$R9{d`K~7X4VQyp~aAkB~V_{}xb09uFAVWb?QcqN0O-W8)LqSGGQ#uMDARr(hARr)fbS`6Ia&m8UUvyz|XJvFCJs@pqZYV=R"
    "Qc_P;UqeAqEFd%>Dj;xWbYEj(W@U3J3LqdLAZ2WGWjYEVARr(hARr)fbS`6Ia&m8UUvyz|XJvFCJs>a&3JPRpW*}r`b7^vAWM61u"
    "Ze(*Pb95{qS1CFQARr(hA|fJ1b98BKVRU66bZ=x~c_(uqcW-iQY;R#?AYpD~AY*TCc4cyOAarjaXlZg~b1otxA_^cNARuI6c_2L?"
    "S1x2>c?uvPARuXGAY@^AAU!=GFggk#ARr(hARr)eWps6NZXh)ZARr(hX=Wf~VR;}uJs>eU3LqdLARr(hAaZ4Nb#iVXF$y3cARuXG"
    "AY@^AAU-`HOhHprUqnG!Itm~lARr(hARuyObairWAUFyjARr)jZ*prOJs>bHFbW_bARumEVQy({VQh0CJs@Lmb#8QDVQy({VQh0K"
    "S1Af0ARr)jZ*prOD?K1?Utw-(ZDDM4ASxg<E-(rpARr)cY+-J6b09q+FbW_bARuOMav*YVcOYqQAXhGQX>4V4Itm~lARr(hARuOM"
    "av*ddX>K5LZ+AKhARr(hARr(hARr(hX=Wg4b6;?5VQzFNbSXLtARr(hARr(hARr(hARr(haBN|2baNmpJs>d(ARr(hcW-iQAS*o}"
    "aBN|2baNmoATcg7H3}dgARr?kaBN|2bZKs9ATJ<db!lv5X>MmAcW-iQ3LqdLAZ2ZEbZKRCAUz;+b!{jyAZBlJAao#UZXk7TY;R+0"
    "Wn^D;X>4V4C|4;UX=WgFX>4U*VRR^0EFg3#AZc?TPH%2yDGDGUARu>da%&(fJs@pqZYX7KaCB*9b1Wb-FexA^ATcg53LqdLAZcbG"
    "WMO$AJv|^eItm~lARr(hARu>da%&(fJs>nN3LqdLAa`MGYiVw0AUz;RNm4~$S3yim3LqdLAZ{Q%AZc!NC~aYMXf9)AX>2HWZ*prO"
    "Dj;`ZY-?$5XCN;iGBPeODJdW=ATbIcARr)VW*}r?c_2JJAT>G)ARr(hARr(hY;P<eXlWolATul=H3}dgARuLIX=Wf~VR;}tJs>wa"
    "3LqdLARr(hAZ%|eAZTeIJs>wMAUO&kARr)RY-wg7WMO$AJUt*eItm~lARr(hARugSEFfrUAUz;4Ff1T3G72CdARuLIX=Wf~VR;}t"
    "Js>hSItm~lARr(hARugSEFfrUAUz;WNlsr#K~6+dEFeumSYJp%PDE1*ARr(hARr(hX=WfXFd#fVAY@^AAUr)FGBqG!Ze$==E^Tja"
    "WqBYzJs>tPFfcj_ARr(hARr(hARr(hXlWolAWcD7Ur0etL{nc<Nkd2qARr(hARr(hY;PbvAZ=-GC~R*mAZTeR3LqdLAZ2WGWjYEV"
    "ARr(hARr)YZ!92aX&^lyIV>PCF$y3cARulaJs@picqnXdEFf)ZZYXGJEFf+vDGDGUARuyObairWAZ`i@3S?zwAaHD9ZeL_!c_?#q"
    "EFf1YItm~lARr<lB12(pY-MC1VRRs9Z*_7YFd!&zav)}Da&vSbbairWAa`XTb7f^9Z)PB2AY@^ADJ~)+A_^cNARu#eE^usNZeL_!"
    "c_2L?S1x2>c?uvPARu*cWMOn=Uvyz|XJvGAD06fyAXh00ARr(hVRLh7XKr6|Z)|0AD06fyAXh00ARr(hb963fX>w(AUw2_{bY)~9"
    "Js@Odb7^vAWM61uZe(*Pb95{qS1Af0ARr)fbS`0TV`y)3b09q+V{dJ6b#!H4VQyn+Z*p@eS1cfNbS`LVa%FR0cVTXHWn?J|ARr(h"
    "b963ZZ*^yAbYFCDWMO$AJs^913JMA%AT2E|EiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElA"
    "EiElAEiElAEiElAEiElAEiElAEeazbYHwm7ZEs{{YziYFEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElA"
    "EiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElA3S(?xb8{d{Z(=$MARr(hUte=<Z*+5CUm!goC?a%eY-J)WAR=L7bZKvHb0RDt"
    "B5-nPZz3!pB5q}6WFjmeB5q}6WM6I~EFdClWqBejAR=^OYh`XCEFdClX>Md9DGCZ8ARr)QWo95>UukY>bYEX6b7gF1EFg4gY-KDU"
    "VPkY@Z*FreAaHVNZ!92gWo2YNPH%2yEFf-WWn^D&JuxgGYiVv|Jt870Itm~lARr(hARu#PY-TQWX>4U6Js@;xY-I`{ARr(hARr)f"
    "Wo%|HVPkY@Z*FrSJs@FYbZKvHa|$3JARr(hARu#PY-TQSa%pcMJs@y$X>ST3ARr(hARr)fWo%|HZe?X;AUz;%Wo2XvARr(hARr(h"
    "b7gF1E^cLIWM6I|Js@snWn^D&3LqdLARr(hAaiAGW-e=Gc_2L?D0FFTWh@|TX>MdG3LqdLARr(hAaiAGW-fGLYh`XAJs?J5Y;$D_"
    "ARr(hARr(hb7gF1E^BFSWFS2tYiVv|3JMBjWo96EVRU73Uw2_{bY)~HV{&hBEFfWLWh@|cY%CyTVR<P!3LqdLAR;0nQ)q8>Y-AvG"
    "XlZjGaBN|2bRc47Aa`MOWpZU?AarkJVR=6yA|eVPARr)VW*}i_WgtC0ATT-#ARr(hARr(ha%FUNa&910a&=`2ARr(hV|5@sAZc!N"
    "D0FNtXJvFKB4clEb7f<7bZK^FUv+MGVRU73Wn>~OATTK@3LqdLAZcbGV|5@tJs>eU3LqdLARr(hAaZ4Nb#iVXRC0A?ARr(hARr(h"
    "ARr(hARr(hARr(hARr(kAa`$dY-Av0X=NaEZ*FO4XmlWNbZBLAcWHBF3LqdLAZcbGV{&hBAU!=GB3DR7K~y3-3LqdLARr(hAaZ4N"
    "b#iVXVP|C^X>K4WGAtl7EFd%~3LqdLAZcbGV{&hBAU!=GB11t^QcqMOItm~lARr(hARuyObairWAYo@^AZczOC^9S{GbsunARr)V"
    "W*}p7Z*U+zJs=`YMNCglB035nARr(hARr)VW*}i_WgtF1AT~M*ARr(hARr(hARr(ha%FUNa&910a&=`OARr(hARr(hARr(hARr(h"
    "ARr?kVsCDBb0BwVZe(wFAT};8F)$z~D={E&WpW^QVRU73Wn>^^VR<PEARr(hARr(ha%FUNa&91DXJsHIATl65Js>a&ARr(hX=Wf}"
    "a&K@TJv|^IQ&dtxS3*TnQduH83LqdLARr(hAZcbGVP|C^K0P2gItm~lARr(hARr(hARuyObairWAYo@^ASEC&AU!=GF(7YpAYo@^"
    "AUr)FISL>kARr(hARuyObairWAYo@^ASEC&AU!=GFbW_bARuXGAY*cGa3DQBAR<&xO+i#oB035nARr(hARr)eWps6NZXi^0b!7@5"
    "ARr)eWps6NZXjW2WgsOWG9W!YATSCF3R6^4L04Z*Nlsr`NkvRVAUz;oYi@61C?ZQoUsF_4L04Z*Nlsr`NkvRVA}k;=DGE?hMNMBs"
    "QcqA{Q$s;aMIb#OUu$k}Vkja@M_*7<MNMBsQcqA{Q$s;aMItO9GA=MF3JMBjWo96AbaG*LVr6o2d0%K@a&~2NbYFL2Zggd2C}C%1"
    "EFgJlWo%?$b#7^Nb1WcaVR<P!3LqdLAR{1ibaG*LVr6o2c_46dX=7y|V{dG1VQ_P0b0BPKZe?L|Y<VDeX>@2Hb9HcVY<W5$b7gF7"
    "AYpSLWnpq`c_3kPAa`XTV_|MA3LqdLAR{1iZy;!4a&~2NbRc(VVRU77WpW^Cb0BYSAarPDAaHD9Zge1bXk~6?c4cxPX>=fIb0BhM"
    "VPttO3LqdLAZcbGd1+;AWM6e|X>@ZSJUt*VAa8OYVP|C^JRmVJItm~lARr(hARuyObairWAVy(qb7cx3ARr)VW*}i_WgtF1ATc%|"
    "Z*m}HVR;}vJs?a$Q&e9>L0KSgav*tWWo%?$b#7^Nb09uFAX8LQL04Z*Nlsr`NkvRVItm~lARr(hARuyObairWAXIX7WeOl5ARr?k"
    "VIXjFZ)A02bZKvHAarkTX=iA3Aa`$dY-Av1cw=Q{WFT~CWguu}Y-Au~VQ?UCW*{^QARr(ha%FUNa&91DXJsI1ZXhTzF)Sc4Gb|u6"
    "H7OurZe$>NX=Q9=Uv+M2baNm+Js>j*3JPRpW*}~5cywQIa&KgHV{~b6ZeL_!c_?&jEFf!XZe%Gs3LqdLAYmXqAVE$^O+idkTWe`<"
    "WL*j%ARr)cY+++%WFS2tX>N2VbZjnXWppSaaBN{?Wn^DuVR<4fATTK@3LqdLAaZ4Nb#iVXaBN{?Wn?TMVG0TgWMyU`Vs&Y3WM67;"
    "Vsj{SbSxlODLM)uARr)WZ(?&GJs?|M3LqdLAY@^AEFfrab#fp*AXhGAVR<YdS1xF8b#e+IARr)RZe(X+ZDk-mAY@^AAU-`HOhHpr"
    "UqnG!3LqdLAaiJCWM6k^Wnpw6Js@drbSPIYb7*B`E@x$QC?Z!#ML|>|EFdr`DGDGUARu#SWn^DwWpZ>NJs@drbSPIYb7*B`E@x$Q"
    "C?ZBhQdCJyNm@lxA}k;<DJcpdARr)eZ)|0AAUz;+bS`plY-Mu_ARr(hZDnk4ZeL_!c_2L?WMO$AK0P2f3LqdLAYyfCY-C?-WoC3B"
    "Js@**a&=>Lb#i5MUv6b(WMyP1b95{qS1Af0ARr)cY+-J6Uw3F_VRT<`a%pcMJs>tVAZcbGWMO$AK0P2YG$3Vcb7dekI0_0NARr(k"
    "AY*TJZge1XWo2Y@AYpc4X>4I)Y-J#3Z*m}TY+-J6X>Ml<ARr(hb7f^@b09q+WNBk`C|53XWo2Y@DGDGUARu9GX>DO_b6;t0Uvp?>"
    "WFS2tdto{tX>N2VS1xmCWn?aAWppTEEFdr`DIjKVav)(KX>K4vPDxEcOjCUdARr(hBOqaJX>DO_b0A}3a&l>9WFTUBAa!nObaNnM"
    "Z*^{TAarkU3LqdLAa`hGVRT<%c4293VPb4$AUz;+Xk}zycW7l{bP6CKARuOCa&%u|c4293VPb4$AUz;+Xk}zyW@U183LqdLAZBlJ"
    "AZd0WX>K4_E@^Idb2<tjARr(hARr)SZ*m}EAZczOK~70cK}=IR3LqdLARr(hARr(hAYpE4ZDDM4UukY%b7*B`TVY)wD?K1-ZgePV"
    "b}naSbSPmgATTK@3LqdLARr(hAa`hGVRT<%c4293VPb4$AS*o}X>N2VX?8AWWppSaS4c%cR3a=OFexbtARr(hARr(hW@U18UtxA("
    "X>4I)Y-J!TJs@drbSP<dE@x$QC?ZBhQdCJyNm@lxA}k;<DJcpHARr(hW^ZyJD0nO&c_}O)a&K&9AZczOa&K&9b1rFgWo>gPDLM)u"
    "ARr(hARr)gY#==#S1xpEY-Mv>d0kt0T?!x|ARr(hARu%gJs>D}EFgI)3LqdLARr(hAZcbGbZj6!Js=`XPeV&ZL?Sv0ARr(hARr(h"
    "ARr(hV{dMBX>N683LqdLARr(hAR{0xEiElAEiElAEiElAEiEk|aBN|2baNmrEiElAEiElAEiElAEiEkyARr(hARr(hX=Wg4b6;?5"
    "VQzFNbZjX)3LqdLARr(hARr(hAY*cGa3DP(bZjnXWppSaV{&hBA}I<WARr(hARr(hARr)NXJsHgAY@^AAT1ziZgePgY%XVIbSNTl"
    "Y+-J6Wn^DuVR<4fAY@^ADJcpdARr(hARr(hARr)lbs#+;X>N2VbZjnXWppSad1+;AWM6e|X>@ZUEFdr`DGDGUARr(hARr(hARu>P"
    "bY*g7WFS2tVsCG3D0FNtXJvFKB6neQWpZU?UvzI|VR<4fAVy(qb7d(h3LqdLARr(hARr(hAZBH9bYFFDbZKlLJs@drbSQLeE@x$Q"
    "C?aNMa&&2IX?kU3Uv+MDX>4C)VR<4fAT2Q|DGDGUARr(hARr(hARu95baNm*AX{AuARr(hARr(hARr(haB^vHAUz;33LqdLARr(h"
    "ARr(hAZcbGV{&hBAU!=GB27h1Pfj8_3LqdLARr(hARr(hARr(hAR{1UZf9w3WjY{8K~h#lQ&b>nb0A?LZf`Aba3Er3W^Zz3AZBTD"
    "b97&MX=Q9=Uu0o<AShvHWgsyyDGDGUARr(hARr(hARr(hARuXGAYo@^AU-`HF)%s`ARr(hARr(hARr(hARr(hARr(hVPkZ2G9W!5"
    "TU`nuARr(hARr(hARr(hARr(hARr)VW*}~FbRc(ObY*g7WFTQ~WFUEUAUq&8AYpD~AYo@^AUr)FF)|=wZe$>CZ*(AKZe(X+ZDl$N"
    "ARr(hARr(hARr(hARr(hARr(hARr(hVPkZ2GA?0oaAj^}C?Z!uR7FxEDGDGUARr(hARr(hARr(hARr(hARu95baOH;VQ_F|Ze%DT"
    "NI_CoMN?EFDGDGUARr(hARr(hARr(hARr(hARuaQVskEGaByXAWGG5+VkmShAYo&4b22O-F)%PJAZuxEWIZBiVRCk5b95ppDGDGU"
    "ARr(hARr(hARr(hARr(hARuFJZggpGb!7@5ARr(hARr(hARr(hARr)VW*}~FbRc(ObY*g7WFTQ~WFU88bY*g1cVTXHWn?I0a&K@f"
    "AYo@^EFg4jEFffIc_}&yARr(hARr(hARr(hARr(hARr(hVPkZ2E@5zRWo~3BB3D6FMN%Ru3LqdLARr(hARr(hARr(hARr(hAaHVN"
    "Zy-G&ZDDvQaB^vHEFd>FAZcbGVP|C^JRmk8Wo&b0AU8QF3LqdLARr(hARr(hARr(hARr(hAZcbGX>N2VbZjnXWppSaV{dMAWn*=8"
    "X?A5_b#8ZIbY*g7WFjmeFexb@K0P2YItm~lARr(hARr(hARr(hARr(hARr(hARusZX>TAsAZ=lIC~$IVZ!91<G9YPYAZTxOav(e)"
    "L|I8rM_))!RZ<{jY;$EGIWZ{;ARr(hARr(hARr(hARr(hARr(hX=Wg3Z*_7YK0P2+Qb$EjR9{F>RZ=<%ARr(hARr(hARr(hARr(h"
    "ARr(hARr(haB^vHAUz;$VR$HTa%pcYAUQHA3LqdLARr(hARr(hARr(hAZcbGVPkZ2Itm~lARr(hARr(hARr(hARr(hARuaQVskEG"
    "aByXAWGG5+VkmShAYo&4b1Wcma%pcYAZuxEWIZBuWo~35DJcpdARr(hARr(hARr(hARr)PZ*FvHZgph}ARr(hARr(hARr(hX=Wf}"
    "a&K@TJv|^IS4c%cR3ad6av)=JZ*U+zJs=`OK~hprR3bVGARr(hARr(hARr(hARr(hZFnF(AVX45P*YoDa&K^5TOw^?cw}LDB3%j~"
    "ARr(hARr(hARr(hARr)VW*}i_WgtF1AZ>UcZ*m|gWo~3=VQpm~VQyp~d37K@ATS_dZe$=~XJsHhJs?9;Pf$}^V{&hBU0Wh%X>xOP"
    "B3&su3LqdLARr(hARr(hARr(hARr(hAYo&4b21=3AX{AuARr(hARr(hARr(hARr(hARr(hcWG{4b97;HbRaz-C~bHkD<Cl`ATKW<"
    "G72CdARr(hARr(hARr(hARr(hARuXGASiBcbRc(ObY*g7WFTQ~WFU8GZeMeBVRCdJJUt*`XJsHfJs@p(AYpD~AbE8lJRn0-Pf$}^"
    "V{&hBU0Wh;VR&D8X=Q9=B3%j~ARr(hARr(hARr(hARr(hARr(hARr(hARr)NZe$>CZ*(9iWo~3=VQpm~VQyp~Xm53LAU-`HGBGJB"
    "Itm~lARr(hARr(hARr(hARr(hARr(hARu95baOH;VQ_F|Ze%DTS3y)oQX(lJARr(hAR{0vF(7qrX>=fBWoB=3Wguu_a&~2NbZKs9"
    "3LqdLARr(hARr(hARr(hARr(hAYo&4b22VraByXAWGEs?K~h#lQ&b`;3LqdLARr(hARr(hARr(hARr(hAaHUZJs>zVAZcbGVP|C^"
    "K0P39cpznLb7demFbW_bARr(hARr(hARr(hARr(hARuXGAYo@^AU+^%csdFoARr(hARr(hARr(hARr(hARr(hARr)cav(h*IW`I)"
    "ARr(hARr(hARr(hARr(hARr)WZ(?&UVQ_F|Ze%D*Z(=BPEFfWHbaOH+AaHUlAZuxEWIZBuWo~35DJcpdARr(hARr(hARr(hARr(h"
    "ARr)PZ*FvHZgph}ARr(hARr(hARr(hARr(hX=Wg9Z*(AcVRU73Wn>^>Ze$>LVRU73Uw2_{bY)~HV{&hBEFfWLWh@|cY%CyTVR<P!"
    "3LqdLARr(hARr(hARr(hARr(hAaHUZJs>wU3LqdLARr(hARr(hARr(hARr(hAZcbGX>N2VbZjnXWppSaV{dMAWn*=8X?A5_b#8ZI"
    "bY*g7WFjmeFexb@K0P2YItm~lARr(hARr(hARr(hARr(hARr(hARusZAUz;BG9YPYAZTxOav(e)L|I8rM_))!RZ<{jY;$EGIWY<#"
    "ARr(hARr(hARr(hARr(hARr)VW*}&9b#fp+Js?$5M@3FlUr0|?QaTDCARr(hARr(hARr(hARr(hARr(hARr)cav(h*ZDDvQaB?gl"
    "IWj2<ARr(hARr(hARr(hARr(hARr(hYHwn5E@5zRWo~3BN^fE)bSxlSB3D6FMN%SNEFf@lEFf!XZe%?obY*U2A}J{fARr(hARr(h"
    "ARr(hARr(hV{dMBX>N683LqdLARr(hARr(hAZcbGV{&hBAU!=GB2!dSL03XWQc_tWItm~lARr(hARr(hARr(hARu95baNm*AX{Au"
    "ARr(hARr(hARr(hARr(haB?6$ATSCbARr(hARr(hARr(hARr)aWo2X_Js?hRZe<D}ARr(hARr(hARr(hARr)VW*}i_WgtF1ATc&N"
    "3LqdLARr(hARr(hARr(hARr(hAR{1bVRLjKaB^>Cbz^jCZ*CxLVQyq)WFTR5AZ2c3Aa7<MVP|C^F*Q0MXkl`8Wpi{OVQyp~V{Bz%"
    "av*eQWeOl5ARr(hARr(hARr(hARr(hARr?kbZKm5AaZGEXmlW9cVT%Tb8jGbXk}q^AY)-}AZBlDY;SiUC~0&ccW-rUWFTZ^V_|t9"
    "X>N3HAYmYPWo2Y33LqdLARr(hARr(hARr(hARr(hAZcbGd37K@ATT-#ARr(hARr(hARr(hARr(hARr(hARr(hYHwn5E@5zRWo~3B"
    "N^fE)bSxlSB1l0}Rz*`(A}k;xL`g>?T_9;@AY@^AAUr)FGB+S)Y;$EGTOvq7QdUJ%R3cq0AUHTIAZuxEWIZBuWo~35DJcpdARr(h"
    "ARr(hARr(hARr(hARr)RY-wg7WMO$AJUt*XH#!O+ARr(hARr(hARr(hARr(hARr(hARr)WZ(?&UVQ_F|Ze%D*Z(=BPEFfDVL`g>?"
    "T`V9rG%O%%X>MdaB4lZ2A}J{fARr(hARr(hARr(hARr(hARr(hV{dMBX>N683LqdLARr(hARr(hARr(hAZcbGb98cHcVcC7a(Q29"
    "VRCk5b97&KVQzF~WGG>0Wh@|hbu1ucVR<P!3LqdLARr(hARr(hARr(hARr(hAYo&4b1q?UaAj^}C?ZHfQdUJ%R3a$~ARr(hARr(h"
    "ARr(hARr(hARr(haB?6$AU8H3X=Wf{XJsHfATc%|Wo&b0AUHS*ARr(hARr(hARr(hARr(hARr(hX=Wg0Ze(X+ZDl$NARr(hARr(h"
    "ARr(hARr(hARr(hARr(haB?6$AUQP(ARr(hARr(hARr(hARr(hBOqpFa&&2IX?kUHAY^ZKVr*q|AarPDAai4AWn^`1Wn>_5a&KgH"
    "V{~b6ZgU_gWo~33Z)PB2XJvCBIV>?TEHN`IF*PX)ARr(hARr(hARr(hARr(hBOrHZX>4U6VPkY@c4Z(aGazJPd2=Z)ARtL(WnpYO"
    "AZBH9bZKm9dSxJQZXjW2Wgs~qVQyp~F*7b8AVy(qY+_+!YjX-9ARr(hARr(hARr(hARr(kAZcbGVIX&DZe(wFAa`MNAZ=-Lb7f>Y"
    "AYo@^ATco@C}VGSWpZ;MF)=JLGbtc1AYo@^ATc!{C}VGSWpZ;MF*PYJ3LqdLARr(hARr(hARr(hAZB_XJs?J5Y;$D_ARr(hARr(h"
    "ARr(hARr(hX=Wg9Z*(AKZe(X+ZDk-~Ze$>4WpZ?1b#8QNY#=-!WMO$AVQyp~W@U18UtxA(X>4I)Y-J!mATT-#ARr(hARr(hARr(h"
    "ARr(hARr(hX=Wf{XJsI1ZXhT*EFdv6DLM)uARr(hARr(hARr(hARr(hARr(hARr)SdMqGja3DP(RC0A?EFeZjQdD11Qb|t=ARr(h"
    "ARr(hARr(hARr(hARr(hWo&6?AYo@^AU!=GF)<)-av))6WgtC0ATc#M3LqdLARr(hARr(hARr(hARr(hARr(hAZB_jAZBnNJs?za"
    "b!99dMnzIoUr<s>ParKIHVPmhARr(hARr(hARr(hARuXGAZB_x3LqdLARr(hARr(hARr(hARr(hAYo&4b1q?UaAj^}C?ZBhQdCJy"
    "Nm@lBDGDGUARr(hARr(hARr(hARr(hARumKWn>^dAR<OZQdCJyNm@lxA_^cNARr(hARr(hARr(hARr(hARuOCa&%u|c4293VPb4$"
    "AT2#0F$y3cARr(hARr(hARr(hARr(hARusZAUz;$VR$HTax5Tba48BPARr(hARr(hARr(hARr)VW*}~FbRc(ObY*g7WFTQ~WFU88"
    "bY*g1cVTXHWn?I0a&K@fAYo@^EFg4jEFffIc_|=aZe$>CZ*(AKZe(X+ZDl$NARr(hARr(hARr(hARr(hARr(hVPkZ2E@5zRWo~3B"
    "B3D6FMN%Ru3LqdLARr(hARr(hARr(hARr(hAaF7uJs>wU3LqdLARr(hARr(hARr(hARr(hAZcbGVP|C^X>K4WIV>PCF)Sc4Gb|u6"
    "H7Pm@ARr(hARr(hARr(hARr(hARr(hARr(ha55l0AU8M)ARr(hARr(hARr(hARr(hARr(hX=Wg4ZgePgY%XVIbSNTYZ*FsCV|8?C"
    "c4c36Zg*jHWpZU?A}k;<DJdX6Js>eU3LqdLARr(hARr(hARr(hARr(hARr(hAaF7uJs>zTAZcbGXm53LAUq&MSxHVuUr0|?QXpk)"
    "b7deoF$y3cARr(hARr(hARr(hARr(hARuXGAZTxOav(lEAXQRFMNU*-NKaK#Itm~lARr(hARr(hARr(hARr(hARr(hARuruAUz;$"
    "VR$HTGAtlDGARlmARr(hARr(hARr(hARr(hARr)cav(h*ZDDvQaB?gla55<hARr(hARr(hARr(hARr(hX=Wf{V{~&m3LqdLARr(h"
    "ARr(hARr(hARr(hAZl-7b1q?UaAj^}C`xZ)D0D0!VPkZ2EFf@lEFf-WWn?{WWo2Y6AZuxEWIZBuWo~35DJcpdARr(hARr(hARr(h"
    "ARr)PZ*FvHZgph}ARr(hARr(hARr(hBOq^dXk~IBV{&hBb08>mZ*5_8Zz&)xEg)xQZe?<5V+tT3ARr(hARr(hARuXGAbE8lJ|Hk4"
    "VQyp~VP|C^K0P2qQcqA*TVrx>a9vv>W@&PBbRu0k3LqdLARr(hARr(hARr(hAZl-7b1q?UaAj^}C`xZ)D0D0!TOvq7QdUJ%R3cq0"
    "AU7~9AZuxEWIZBuWo~35DJcpdARr(hARr(hARr)RY-wg7Zf|rTcVTp8a%E&7VQyp~Zf|rTWo~3=VQpnP3LqdLARr(hARr(hARr(h"
    "AZl-7b1q?UaAj^}C`xZ)D0D0!TOwCMR7FxET`V9sG%O%%X>MdaB6MYLWFjdk3LqdLARr(hARr(hAY*TCbZKsNWeN%)ARr(hARr(k"
    "AT2E|EiElAEiElAEiElAEg)fTX>DO_b094(EiElAEiElAEiElAEiDQlARr(hARr)VW*}%`b6;U@X>DO_D0FNoItm~lARr(hARr(h"
    "ARudLZe$=mAaralXJvFKB4KW6ZDDL8DGDGUARr(hARr(hARu8NJs?3&NligaQ(J3kZe(2wARr(hARr(hARr(haBN{?Wn>^dAZc!N"
    "D0FNtXJvFKB5-VBV`XGtWMO$CEFdr`DGDGUARr(hARr(hARu{lAUz;yZgePgY%XVIbSNTuX=Q9=Uv+M2baNssATTK@3LqdLARr(h"
    "ARr(hAZBG`AUz;rZ*OcUbZjnXWppSaW@ThwbZ=x~c_J(zMqzAoWhp5NARr(hARr(hARr(hV_|Y-WFS2tVsCG3D0FNtXJvFKB4c54"
    "Wn^D;Z)9P4A}k<AVQh0{DJcpdARr(hARr(hARr)cWo~33Js@drbSQLeE@x$QC?arWZe(e0XJ2Dsa%Ep)Z*FySA}k;<DJcpdARr(h"
    "ARr(hARr)SWpZ?1VRj%rAYyNCY$$YWE@x$QC?aNMa&&2IX?kUHUtxA(X>4I)Y-J)WAVy(qb7d(h3LqdLARr(hARr(hAY*kPJs@dr"
    "bSQLeE@x$QC?aETZgXX0b#!TVWnXn}W@Tg|EFdr`DGDGUARr(hARr(hARr?kW_5IRa%CWJa&KgHV{~b6ZXje~d2=8rWo~3GZ)PoI"
    "VR;~BDIhu^Wh*ftEg*1gVPj=vAT1zfX>xOPAU!=GYbt4ObY*gOVQefQWgt8~ATl@#ARr(hARr(hARr(haB^>CUvzJ7X=iA3AUz;P"
    "VQh0{3LqdLARr(hARr(hAZB%Rb#i52aB^>CAUz;PVQh0{3LqdLARr(hARr(hAZBlJAY~wFZXj}DZf9jEWMO$MATl{AItm~lARr(h"
    "ARr(hARr(hARuIOAUz;uAS)m-AT1zpY+++%WFRddVOt_*X>xOPB3%j~ARr(hARr(hARr(hARr)VW*}s9AU-`HFd$)WWFTa7ASEDS"
    "TOw(0bY*gOVQeB@AU!=GFggk#ARr(hARr(hARr(hARr(hARr)VW*}uCJv|^~VR<?VARr(hARr(hARr(hARr(hARr(hARr(haB^>C"
    "UvzJ7X=iA3AUz;da&=`2ARr(hARr(hARr(hARr(hARr(hW_5IRa%Ep|a&KfHJs?zab!7@5ARr(hARr(hARr(hARr(hARr)Oa%Ev_"
    "3LqdLARr(hARr(hAYo&4b09q+TU`nuARr(hARr(hARr)cav(h*FbW_bARr(hARr(hARuXGAZ2c3XJKt+Itm~lARr(hARr(hARr(h"
    "ARuXGAbE8lJ|HkU3LqdLARr(hARr(hARr(hARr(hAZl-7b1q?UaAj^}C`xZ)D0D0!TOvq7QdUJ%R3cq0AUQQGAZuxEWIZBbZfR{{"
    "Y$7Qs3LqdLARr(hARr(hARr(hAY*TCbZKsNWeOl5ARr(hARr(hARumKWn^DwWo2X_Js>D<Z*(ALWn?KJVQyp~C}d%IAUq&4I3RCw"
    "AaHVTWM6b|ZfR#|bSWTVZe$>2VR;}vJs>d(ARr(hARr(hARr(hX=Wf~VR;}uJs>hTAYpD~AZ~ATAZBG`AYpD~AZ~ATAaHVTWM6b|"
    "ZfR#|bUF$kARr(hARr(hARr(hARr)aWo2YvW@Tk$AUz;PVQh0{3LqdLARr(hARr(hAZcbGZe?X;UuI=xWFTQ~WFU8FWnpw*VRm6@"
    "Y++(-Wgt8~ATT-#ARr(hARr(hARr(hARr(hZe?X;UuI=xWFS2tMqzAoWgs9RARr(hARr(kAZ~ATXlZU|AarjaW@Tk$Aa`kWXdrTF"
    "XJ~XFZf|!AARr(hARr(hARr(hX=Wg9Wo2YvW@Tk$Itm~lARr(hARr(hARr(hARu95baO6YaByXAWGEs=MMXp+DGDGUARr(hARr(h"
    "ARr(hARusZAUz;BFbW_bARr(hARr(hARr(hARuXGAY*kPK0P2YItm~lARr(hARr(hARr(hARr(hARusZAUz;BHVPmhARr(hARr(h"
    "ARr(hARuXGAZTxOav(lEAVx(+L|;`>M@3FlUr0|?QaTDCARr(hARr(hARr(hARr(hARr)cav(h*ZDDvQaB?glIWQ>-ARr(hARr(h"
    "ARr(hARr(hX=Wg3Z*_7YK0P2+Qb$EjR9{F>RZ=<%ARr(hARr(hARr(hARr(hARr(haB?6$AZ=lIC~$HtAUQKB3LqdLARr(hARr(h"
    "AZcbGZf|rTV_|Y-WFTQ~WFTgBbairNUvP47WFTQ~WFT;5Ze$=lAT=OiZe$=RW@Tg`Z*m}RWo2YvW@Tk$DIj5PWFTZ=c_2I>GB`R4"
    "ARr(hARr(hARr(hARr(hVPkZ2E@5zRWo~3BB11t^MItE*ARr(hARr(hARr(hARr(haB?6$AZ=lIC~$HtAVWb?MPE=-Nlz&XARr(h"
    "ARr(hARr(hX=WgKbs#<<Fggk#ARr(hARr(hARr(hARr)NV{~&aVQ_F|Ze%DTNI_CoMN?EFDGDGUARr(hARr(hARr(hARusZAUz;$"
    "VR$HTax5T0PDxEcOkYSrQdUJ%R9{e1Nlz&XARr(hARr(hARr(hX=Wg1WpZ?1VRj&4Ze$>2VR;}tAWT71R9{3vSvm?JARr(hARr(h"
    "ARr(hARr)NV{~&aVQ_F|Ze%DTLr+XhMMG3yMnzIoNlZyvMN%Ru3LqdLARr(hARr(hARr(hAaHUZJs@picqnjkEFeQqOiV>XR9{e1"
    "Nlz&XARr(hARr(hARr(hX=Wf{V{~&m3LqdLARr(hARr(hARr(hAZ}%4WOry~VRRrpAR<OZMMNSXX>K54V{~&0ARr(hARr(hARr(h"
    "ARr(hYHwn5E@5zRWo~3BN^fE)bSxlYV{~&YAaHUlAZ}%4WIZBRNJT+ZA|PpIAZ}%4WOry~VRRs6Y;$EGPH%2yEFf!XZe%?oVQy({"
    "VQeBPDGDGUARr(hARr(hARuFJZggpGb!7?)ARr(hARr(hBOom;EiElAEiElAEiElAEiE8rZE$pXAaitbbz^jOa%CVbEiElAEiElA"
    "EiElAEiEkyARr(hARr(hX=Wg4b6;g`aCCWJb98cbV{|BVY$-YlARr(hARr(hARr(hYiVv|Uv6b(WMyO^Js=`OPgf!!X=WgFY%XVI"
    "bSNTgX>Md9DIh&PAR<sfQ&d$_MIs<&Y;$EGB1cb8Q$->QARr(hARr(hARr(hBOqgFZ*OyDAa`hKV`v~@ZfR{{Y#?-RAaHD9V`VxZ"
    "aB^j4WpW^QXlY|;Wp-t9Aa`XTXkm6`3LqdLARr(hARr(hAZBlJAY)-}WFTp7AShcRLr+&CEFdCNNJT|ZB3&S9W*~HIE@x$QC?acV"
    "Ze$`UAU!=GB2Yn7R8>+%A|PdKb7de~B1cb8Q$-?ODLM)uARr(hARr(hARr(hARr)VW*}j1X>DO_b6;t0Uvp?>WG-iAbSPtCZe%PV"
    "FexBDATT-#ARr(hARr(hARr(hARr(hARr(hVQy({VQh0>X>MP0Xk}zuV_|M&T_7zzATbIcARr(hARr(hARr(hARr(hARr)WZ(?&U"
    "VQ_F|Ze%D*Z(=BPEFfDVP)tEXMItO9V_|M&T`V9sGAtl&Wo2YNV_|M&EFf!XZe%?oaBN{?Wg;ml3LqdLARr(hARr(hARr(hARr(h"
    "AYyW5VQUH?ARr(hARr(hARr)RY;$Eg3LqdLARr(hARr(hARr(hAZcbGa&K&9AR#><B0)|`O+ic|AYpD~AZ~ATAZ2c3XJKt+AYpD~"
    "AY@^AAUq&4H#!O+ARr(hARr(hARr(hARr(hARr)WZ(?&UVQ_F|Ze%D*Z(=BPEFfDVL`g>?T`V9nFf1TzX>MdaB4lZ2A}J{fARr(h"
    "ARr(hARr(hV{dMBX>N683JM?~ARr(hARr?kEiElAEiElAEiElAEiElAAa`YDWOE=bEiElAEiElAEiElAEiEkyARr(hARr(hX=Wg4"
    "b6<C5Wn?IHY$-YlARr(hARr(hARr(hX=Wg9Z*(AKZe(X+ZDk-~Ze$>2VR;}tJs>hSAYpD~AaZYPWguy8ASfbNNJT+ZA}k;xQ&dtx"
    "S0XGRB0)|`O+ic|EFdCHMNCglA}k;xLqSqfPgEi)Itm~lARr(hARr(hARr(hARuaQVskEGaByXAWGG5+VkmShAX_3tNk<}GEFd;8"
    "AZcbGC~|LXWgtC0AR<##QbAWDAYpD~AUGgAJs@OZc_2JJATc#5AZ2WGWgsXvG9YPYAY@^AAU-`HF*G1$Y;$EGH8LqIAZuxEWIZBe"
    "X=frSDGDGUARr(hARr(hARuFJZggpGb!7?)ARr(hARr(hBOom;EiElAEiElAEiElAEiE8rZE$pXAarSLWgsmrEiElAEiElAEiElA"
    "EiDQlARr(hARr)VW*~HIAZc?TPH%2yItm~lARr(hARr(hARuXGAZ2c3XJKt+Itm~lARr(hARr(hARr(hARuFJZggpGb!7@5ARr(h"
    "ARr(hARr)VW*~BJY-J!lJs=`MPDxEcOd>i8ARr(hARr(hARr(hARr(hBOqdRX>4R5VIXi}b98lbWgu^EY<VDNZ*m}EZfR{{Y;z!V"
    "Xkm09VRB_4Z*CxHVQyp~Z*m}EVsCYHAarjEARr(hARr(hARr(hARr(hBOqdBAYyNIXJ~XFC~jq8a%FRLAZBu9WgvQQZe<{JX>4V4"
    "AZBTDb95jrEg*7lY-MvGVRB_4Z*pX1a%E&G3LqdLARr(hARr(hARr(hAZcbGVs&Y3WM6D$W^^DvATT-#ARr(hARr(hARr(hARr(h"
    "ARr(hVs&Y3WM6D$W^^DeJs>d(ARr(hARr(hARr(hARr(hARr(hYHwn5E@5zRWo~3BN^fE)bSxlSB0^P3OhjK$K~q#!Qbi(NEFd;E"
    "EFf!XZe%?oVs&Y3WFjdk3LqdLARr(hARr(hARr(hAZ2W6W*{hJVR;}tJs>e4Z*m}HVR;}vJs>eSDIj5PWFTZ=c_2JJAXi95K~!H%"
    "K~q#;P)tEiR3Kq)WFT{8Wn^<MXJvFKB3DR7K~y3vATTK)J|HkU3LqdLARr(hARr(hARr(hARr(hAaiA9WOG|0S4c%cR3cp<Ej=JH"
    "3LqdLARr(hARr(hARr(hARr(hAZl-7b1q?UaAj^}C`xZ)D0D0!TOv?QK~7X6EFdCRNJT+ZB3&#XaBN|2bYFL9Wnpw*aB^vHEFf!X"
    "Ze%?oaBN|2bRsD!3LqdLARr(hARr(hARr(hAY*TCbZKsNWeOl5ARr(hARr(hARuXGAaZYPWgtC0AR<jgOixZCItm~lARr(hARr(h"
    "ARr(hARuXGAY@^AAU!=Gb963kWo&P5UvO+;ZggK{VR;~7Ze$>HWo2Y@E@x$QC?ZWoOixZCEFdr`AU+^4Itm~lARr(hARr(hARr(h"
    "ARr(hARu#PWn^<(B27h1Pfj9TAT2#0F$y3cARr(hARr(hARr(hARr(hARuaQVskEGaByXAWGG5+VkmShAX_3(OhHamA}k;xO+`#k"
    "P9j|_AU7~9AZuxEWIZBqY+-J6A}J{fARr(hARr(hARr(hARr(hWo&6?AY@^AAU+^-bS`aWY;SH~aBN|2bYEm)c_3kKWFT{8Wn^<M"
    "XJvFKB3DR7K~y3vATTK)J|Hk4VQyp~WMO$AJUt*+NJT+ZUra$$R9{d`K~7XU3LqdLARr(hARr(hARr(hARr(hAaiA9WOG|0S4c%c"
    "R3cp<Ej=JH3LqdLARr(hARr(hARr(hARr(hAZl-7b1q?UaAj^}C`xZ)D0D0!TOv?QK~7X6EFdCRNJT+ZB3&#XaBN|2bYFL9Wnpw*"
    "aB^vHEFf!XZe%?oaBN|2bRsD!3LqdLARr(hARr(hARr(hAY*TCbZKsNWeOl5ARr(hARr(hARuXGAaZYPWgtC0AR<##QbAWDItm~l"
    "ARr(hARr(hARr(hARuXGAaiA9WOFWOWppSaQ&dtxS3*TnQduG_ATTK)J|Hk4VQyp~WMO$AJUt*&R8m1#Ura$$R9{d`K~7XU3LqdL"
    "ARr(hARr(hARr(hARr(hAaiA9WOG|0Q&dtxS3*TnQduHhAT2#0F$y3cARr(hARr(hARr(hARr(hARuaQVskEGaByXAWGG5+VkmSh"
    "AX_3(OhHamA}k;xQ&dtxS3*TnQduHhEFd;GEFf!XZe%?oaBN|2bRsD!3LqdLARr(hARr(hARr(hAZ2W6W*~EAWn^<MXJvFKB3DR7"
    "K~y3vATTK)J|Hk4VQyp~WMO$AJUt*+NJT+ZUra$$R9{d`K~7XU3LqdLARr(hARr(hARr(hARr(hAaiA9WOG|0S4c%cR3cp<Ej=JH"
    "3LqdLARr(hARr(hARr(hARr(hAZl-7b1q?UaAj^}C`xZ)D0D0!TOv?QK~7X6EFdCRNJT+ZB3&#XaBN|2bYFL9Wnpw*aB^vHAT1y="
    "EFf!XZe%?oaBN|2bRsD!3LqdLARr(hARr(hARr(hAY*TCbZKsNWeOl5ARr(hARr(hARuXGAaZYPWgtC0AR<FSQc_P;B035nARr(h"
    "ARr(hARr(hARr)VW*~EAWn^<MXJvFKB11t^QcqMOEFdr`AU+^4AYpD~AY@^AAUr)FLqSqfPgGw_K~q#;P)tEiR5}VEARr(hARr(h"
    "ARr(hARr(hARr)fWo2Y@TOvb2Qc_P;B3&RYJs>d(ARr(hARr(hARr(hARr(hARr(hYHwn5E@5zRWo~3BN^fE)bSxlSB2Y|0PE;Z+"
    "AR<FSQc_P;B3&#XaBN|2bYFL9Wnpw*aB^vHEFf!XZe%?oaBN|2bRsD!3LqdLARr(hARr(hARr(hAZ2W6W*~EAWn^<MXJvFKB3DR7"
    "K~y3vATTK)J|Hk4VQyp~WMO$AJUt*+NJT+ZUra$$R9{d`K~7XU3LqdLARr(hARr(hARr(hARr(hAaiA9WOG|0S4c%cR3cp<Ej=JH"
    "3LqdLARr(hARr(hARr(hARr(hAZl-7b1q?UaAj^}C`xZ)D0D0!TOv?QK~7X6EFdCRNJT+ZB3&#XaBN|2bYFL9Wnpw*aB^vHAT1y="
    "EFf!XZe%?oaBN|2bRsD!3LqdLARr(hARr(hARr(hAY*TCbZKsNWeOl5ARr(hARr(hARuXGAaZYPWgtC0AR<>tML|>|Itm~lARr(h"
    "ARr(hARr(hARuXGAaiA9WOFWOWppSaS4c%cR3a=OFexBDATS_dZe$>2VR;}tJs?*|ML|?wOhHprUr<azPE<MyARr(hARr(hARr(h"
    "ARr(hARr(hb7f^@b6X-;NJT+ZB3&RYJs>d(ARr(hARr(hARr(hARr(hARr(hYHwn5E@5zRWo~3BN^fE)bSxlSB2Y|0PE;Z+AR<>t"
    "ML|>|T`VAQY+-J6Uw3F_VRT<`a%pcYAZuxEWIZBqY+-J6A}J{fARr(hARr(hARr(hARr(hV{dMBX>N683LqdLAaZ4Nb#iVXYHwn5"
    "3JMBjWo966cWz~5Utw-(ZDDM4C|4{XYiVv|DLM)uARr(jA|g<1VPj=vAS)ngZXk1LWn>^LAY);2a%p8`E+Qf#3LqdLAZ{Q%AY*TJ"
    "ZggK^ZfR{{Y;!1AEFf!XZe%GSD<ElZbSPIYb7*B`E@x$QC~IkMWGo;sDJcpdARr)SZ*m}Mb|7hPAXhGFZgz7z3LqdLARr(hAZ{Ql"
    "Js@drbSP<dE@x$QC~IkMWGo;sDJcpdARr)eWps6NZXj+73JPRpW*~EPa&=>Lb#i5MUv6b(WMyP1b95{qS1CFQARr(hA|fJ4Z+9SV"
    "VQzUKZEtdAAaG%GbairNb0BwRAaiJMb!=oHXkm6`AaZ44WO+Iub#8EMVPj=vAYpE4ZDDM4AS)nYZfR{{Y;y`AARr)jWgv5OX>4pD"
    "X>N37Ze$>IZy;iIc`P7pX>N6MAZ2ZEba^0ibaHiLbairNb0A@Ca%Ev;c_4FiVQyq;Zf7ncA|eVPARr)VW*}ECWMO$AJ|I&_MMY3w"
    "OhHprUqV$`Itm~lARr(hARuyObairWATSCbARr)PZ+CMbJs@v)Ze?U&VQy({VQh0KS1ceRLr+&CDGDGUARu#SWo2+6Js@v)Ze?U&"
    "VQy({VQh0KS1ceRQ%FTcP$DS`ARr(haBN{?Wn>^dAY*TJZggK^ZfR{{Y;!1ADGDGUARu*aaBN{?Wn>^dASh#RcXJ>sAaiJCWpF7V"
    "Eg*1gVPj=v3LqdLAZ2ZEbZKRCAUz;33LqdLAZBlJAaZYaAZczOS1xpEY-Mvg3LqdLARr(hAZBlJAao#UZXj}RcRC6nARr(hARr(h"
    "ARr)VW*}*EUuA7@ba`KMbaHiLbSQKwItm~lARr(hARr(hARr(hARuLJaCB*9b08}{ATbIcARr)jVQzFFJs@picqlL|AaissV{dm~"
    "bYXI5Wpp4dAY*TLb15JzAZ=lIC@?G_b963qXk}$^Uvyz|XJvFCEg*AfWo2+F3LqdLAaZ4Nb#iVXZDDvQFf1T-Zg6a2V`XF@D<F4a"
    "Zge0mAZ2ZEbZKRCDGCY-WMyU`cVTXHUu|!4WnXi2a&=>Lb#i5MD06fyAXh0m3LqdLAaZ4Nb#iVXb98cbV{~<LWpiI{Wo2Y#WGHiV"
    "EFf1YAU+^43JMA%AT2E|EiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElA"
    "EiElAEiElAEeazbVRLh7XKrm}ZgdJGAT2E|EiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElA"
    "EiElAEiElAEiElAEiElAEiElAEecgqM@3FlUr0|?QXoAbUu$k}Vkja@M_*M^M@3FlUr0|?QX(uMF*PX)MnzIoUr<s>Par)YUu$k}"
    "Vkja@M_)!oQdD11Qb|uDEFd^EDGEbDQbk`-Qb|uBJs@9eZf{~JB1=bKLqSqSUr<s>Pa-TJH!vv*K~70cK}=srK~h#lQ&eA2Qb|uB"
    "Js@9eZf{~JB1=bKK}cUvQb|uDEFd;GDGEbROiV>XR9{e1Nlzd>AYW^4Z(=ASOGjTrPfScjLsVZ-Qb|uDEFd;ADGF0XOiW)$PgPQ1"
    "Pfko(AUz;oYi@61C?ZQoUsFX)OkYS(RZ?G1PE1)MEFdr`3R6W)OkYS(RZ?F}K~q#9Js@9eZf{~JB1=bKQ$<WnUr0|?QeR9#Q&b`>"
    "ATTKkMny$LUsX~^MNU*-NKaK#AUz;oYi@61C?ZQoUq(emL|;`>M@3FlUr0|?QX(uMF)=9$O+i>+NI_0SQy@JcUu$k}Vkja@M_)}r"
    "SYJp%PDE28EFdv6DGEqQQbk`^K}<^^Js@9eZf{~JB1=bKNJ&yfUspj)OCl^FGA=PG3Qb8)Ur0etL{lI=AYW^4Z(=ASOGjT#Nlsr#"
    "K~6+dA}k;=GARm8L0Df%K~6+dUs6d!NFY5RUu$k}Vkja@M_)}rSYJp%PDE2*Qb|KdA}k;=G${&1SxHVuUr0|?QXoAbUu$k}Vkja@"
    "M_)u)Nlr&!NKaK#A}k;=GbsvJNJT+ZUra$$R9{d`K~7X4Js@9eZf{~JB1=bKS4c%cR9{R%Q&b`>ATl;73PM#(OJ77%Pf#E|AYW^4"
    "Z(=ASOGjTqRZL4?L{d*sA}k;<DGE<kMN&pgPgh@4MNCW}Js@9eZf{~JB1=bKPgX@zModpvA}k;=DGFCXOiN!wPg7JNJs>hQE-(sH"
    "R7pciUqVk#RZ}26AT}^AFbYpjUsOp<MPEWsPE}JNJs@9eZf{~JB1=bKPflM{NlZl|EFd#AFfcAKDGF3cMN(f!K~Nw)ATuy9FfK3("
    "P*O=xUsoVKAUG~C3JMBjWo961Z(?6`X=QRKaB^vHDLM)uARr)VW*~5KX>TAtJs>$SItm~lARr(hARuyObairWATtUeARr)VW*~5K"
    "X>TAtJs>tPItm~lARr(hARuyObairWATkOdARr)eWps6NZXhuV3JO6^Lr70jUsoVKAYW^4Z(=ASOGjTpPD4mfQeRghEFd&4FewTO"
    "3S?zwAY*TBaCLNLUtw-zXm4_JC|4{XZeM6&Ze(*QItm~lARr<lB2#c|X>=fTXk{RGZ*ps8WFT~DY-MvGX>N3HAYpE2b!=gBAaiA7"
    "bZ>HVAYpQEb#7!JbZBKDb7*B`AYpD~AZKZIWeOl5ARuL7V`v~~X>w&`AZTH3WFT~CWguf^Zgg^QX=EU8W*~2FWgv59V{~tFAYpSL"
    "X>@ZSbY*gKX>@OLc_3kKV`y)3E(#zZARs76VQyp~X>Md`V`Xz7VRB_4F)l7{Uua=&WOF+pbZBKDW?^z|WpW^MbYXdOAYmY8a%E*8"
    "VP|D-bS^0(A|eVPARr)NZewU~a&sU(AbWiZARr(hX=Wg9Uua=&WOE=qJs>b3Z*m|(PD4mfQeRgfJUt*VItm~lARr(hARuyObairW"
    "AYpD}Xm4_J3LqdLAaHbZAUz;kT?!x|ARuOMav*siX>K5LVQyz-C_+y`QbZ{_3LqdLARr(hAZBlJAb22YZXj}DZf9jELQg?bL@7E7"
    "ARr(hARr(hARr(hbZj6!AXhGQX>4V4TX|hucwGu0ARr(hARr(hARr)VW*~HIAU!=GB1}(1OGQK?Itm~lARr(hARr(hARr(hARuFJ"
    "ZggpGb!7@5ARr(hARr(hARr(kAarSLWpf~OXkm09cWG>FAZ}%4WFU5Fb7^#QIv{XtVQzGDAYpD~AYpE4ZDDM4J0N9kaCCVfbZKm5"
    "b0A?LVrg^=ARr(hARr(hARr(hcOX3=F)lC)ARr(hARr(hARr(hX=WgFY#?cKAWm;?WjYEVARr(hARr(hARr(hARr)jAUz;3E;BU>"
    "ARr(hARr(hARr(haCCDnVQ_F|Ze%DZZDDk1E@5<GZZar&AT1z6MNU*jQd==yEFgFwEg(ZhPE<uwTQFTIEFgF+AbBhxcPS|fARr(h"
    "X=Wg9Z*(AVbaOfiARr(hARr(ha%FUNa&91DZewU~a&rnGARr)cbaO6qZ*p`fDGDGUARu&abYW~?cOX3=b9HSfa9cB7AZBlJAaEdQ"
    "ZXj@Ub14cSARr)cWpW@rAarkZVQgP_ATJ<pUua=&WOE82ARr)NV`CsaATTa43LqdLAZK!LAUz;kT?!x|ARuRHAUz;43LqdLAZBlJ"
    "AaEdQZXj@Ub2<tjARr(hARr)Ta&RtTaByXAWGHYc3LqdLARr(hAYo%;AS*o}a9cB73LqdLARr(hAZcbGVPj(;K0P3CWpW@YAZKYH"
    "VQyp~XK5fjJs@shXkl(-b2<tjARr(hARr(hARr)fcpyC>b9HSfaa%E6ASxhnTQglCW^ZyJaUf}KAZK!LDIhN(ZDDvQF=Z_|EFg1r"
    "Z76YDGhHBNZ*m}UAZczOXL4{UDGDGUARr(hARr(hARu#jAUz;+b!{kdTQXfBDj;!NGhHBNZ*m}UAZczOXL4{UATJ<oVR$GpWi2@@"
    "AaiwXC~;deT_9#}av*UaX>K5Aa&Rdr3LqdLARr(hARr(hAYpD}Xm4_JTW4uqAUz-`b9gKub9pHWARr(hARr(hARr(hXK5fSJs>d("
    "ARr(hARr(hARr(hXL4{LJs?|M3LqdLAZcbGXL4{LVQyp~XK5fjJs@shXkl(-b2<tjARr(hARr)fcpyC>b9HSfaa%E6ASxhnTQglC"
    "W^ZyJaUf}KAZK!LDIhN(ZDDvQF=Z_|EFg1rZ76YDGhHBNZ*m}UAZczOXL4{UDGDGUARr(hARu#jAUz;+b!{kdTQXfBDj;!NGhHBN"
    "Z*m}UAZczOXL4{UATJ<oVR$GpWi2@@AaiwXC~;deT_9#}av*UaX>K5Aa&Rdr3LqdLARr(hAYpD}Xm4_JTW4uqAUz-`b9gKub9pHW"
    "ARr(ha%FUNa&91DZewU~a&rm_3S?zwAYpTJX=iRIb95{qS1cfEZ(?&PItm~lARr<lB2r~^b#iVXY-w|JAa7<Mb#7^NAYo&4X>V?G"
    "ASh{WWMz0DVQgt<Ze?U3cWHEJAXhGRZfSILDJ~)+A_^cNARumEb#7^Nb09q+Y-MgJS1xsKX>@Zb3LqdLAYo&4X>V?GAUz;kTOv?F"
    "Q&S>cAZBlJAYUM9ZXj}DZf9jEZeMk7X>@ZbT?!x|ARuI6c`P7kZ*_7YJs?*uWMO$MAXhGEZ*_7CARr(hWo~3=VQpm~Js@OZc_2PL"
    "AWT71R9{3vSqdN^ARu&ga&B{9Y-MJ2UvzI|VR;}uAXHF9AT1y<AT1zhZ*_7YARr(hARr(hARr(kAar$dZgU`EW^`q8AarPHb0BYS"
    "Wgv8KWMO#<ARr(hb7*B`AUz;tX=8LKS1xmCWn?J|ARr(hb7*B`Uvh76Z6G}$Q%FTbUqeAqAT1zsXk}zybZ>NFY$#VL3JM?~ARr?k"
    "aAk5Wb#7^NAZc!PWo~qDa%p9AASi8hbYWs_Wgug3aA{?8AZBlJAaZ4MWpZ|5bZKvHDGDGUARuXOc5@&-AX{W<V{|BKb}1lcZ*m}M"
    "b|7hPAXhGFZgz8B3JM?~ARr?kEiElAAaihKV`*V*Iv`|nZ*U-LZ(?&GW^ZyJb#7^Nb0A}3a&mcTZf78NVQh6_Vr*p~b98lPW*~2J"
    "AZ2c3XJKt+AT2E|EeaqYARu*aX>?y^Z*pU0WFS2tdwmKZARr)SZ*m}MEFf@ib0BGMAZ2cKZDn#{bY&=4E_H5cbaN>>3LqdLARr(h"
    "AZd0WJs@drc5_>4T?!x|ARr(hARu;OY#==#FfK3(ARr(hARr(hV{UXHJs>a&ARr(hARr(hW^ZyJX>=?gZXjuHAZd0kX>?_6b0{e~"
    "3LqdLARr(hARr(hAZcbGX>=fIZXiuTQcFctUr<3(K}}O2VQyp~ZXiA&Fggk#ARr(hARr(hARr(hARr)iVQe5PJs@r%Dj-)baB^v5"
    "WpgfPWppTMbSxlEK~hUaR9{d*QbA2qTWNG%TOwj%b7dl3DGDGUARr(hARr(hARr(hARuFIbRa7|AZ`jEARr(hARr)fWGo<Lb7&wv"
    "AZ}%0a%FRLUvp?>WGHZNb14cSARr(hARr)VW*}p3bRa!FATT-#ARr(hARr(hARr(hV{dMBX>N683LqdLARr(hAZ>MXbYEn0Z*U+z"
    "AVy(qb7cx3ARr(hARr)VW*}v5WM^S*WjYEVARr(hARr(hARr(kAZ>MXbRcqNVPj|@bZBKDb7*B`AYpD~AY^iHa3Er3W^Zz3AarPD"
    "AZ%fCbRcDTWn*=8Wn>_8bY*Y~ARr(hARr(hARr(hX=Wf~b7&wwJs?*ub97~Jb6;#_W^^DeATc@$ARr(hARr(hARr(hARr(hZFO^W"
    "Uu1G`a3DP(RC0A?3LqdLARr(hAZ=xAZ*FrSJs@dzE@x$QC?ZWoOixZCEFdr`3LqdLARr(hAZcbGZDnk4ZgU_$Js>tZ3LqdLARr(h"
    "ARr(hAZ>MXbYEn0Z*U+zAXIX7WeOl5ARr(hARr?kaB^jBX?1NNXK!z0b2=bmVQy<6bZBL5AaHVTZE$pKc_4FdAarPDc_3kOWgv5J"
    "Y-AvGZ)9P4AShvIWnp9>Z)OT0ARr(hARr(kAa8JRZ*FC7baNngXm22MWo&F9bZBKDZe@6MAZ>4QZfS03DGDGUARr(hARusZWo=(}"
    "VQe5hAaiwXC~0;sXJvFKX>=?gFexA^AXhGMa%p2_b1r9PbSP<bEFeumQcFctUr<3(K}}OzX>?s%B4S~4Wg=ZEAZBlJAZc_UX>K4;"
    "QbkQkRZS@hARr(hARr(hX=WgBa%F8_c42HFK0P2aFfbr0AW%|8O<zP(Pf%Y|LqSYMAYpD~AY^lBAUr)FGCB$%ARr(hARr(hARr)Z"
    "b#ruIWO8qCAUz;da&=`2ARr(hARr(hX=WgBa%F8_c42HFK0P2gFfbr0AW%|8O<zP(Pf%Y|LqSYMAYpD~AY^lBAUr)FG&%|(ARr(h"
    "ARr(hARr)Zb#ruIWO8qCAUz;da&=`2ARr(hARr(hX=WgBa%F8_c42HFK0P2ZFfcG6Dj-l&MNMBsQcqA{Q$s;aMId2rWFTa7Xdpa2"
    "AU8S+ARr(hARr(hARr(hZFO^WUu1G`a3DP(RC0A?3LqdLARr(hAR{1Rb!=-oAY^YKZf|rTcVTR6AYpQEb#7!JcWHEJAYmY4X=fm8"
    "Z((F0C~0&cVQg$5Y+-I>b0BGMAarPDAaiJCWFTR53LqdLARr(hAR{1dX=iA3AYpD~Aa8bMa%OCAcOYqVAY^58baHQbWn?=bb7gF7"
    "X>MmAWnpq`X=QRCX>%ZAWps39av))Dd3RxXDGDGUARr(hARuXGAVO74OJ77%Pf$7vARr(hARr(hARr(hX=Wf}Zge0%Js>zBVQyp~"
    "WOHaBJUt*XItm~lARr(hARr(hARr(hARujZb97&1a&K@TJs?zab!7@5ARr(hARr(hARr)VW*}p3bRa%GATcx`VQyp~WOHaBJUt*Z"
    "Itm~lARr(hARr(hARr(hARujZb97&1a&K@TJs?zab!7@5ARr(hARr(hARr)VW*}&9b#fp+Js>hLAYpD~AY*QHAU-`HG$3JaWFTa7"
    "Xdpa2ATv4&ARr(hARr(hARr(hARr(hZFO^WUu1G`a3DP(RC0A?3LqdLARr(hAZcbGXm53LAU-`HGBO}xZe$>2b7&wuJs>e4VQyp~"
    "b7*B`Uvh76Z6H1%V{UXh3LqdLARr(hARr(hAZ>MXbYEn0Z*U+zAXIX7WeOl5ARr(hARuXGAZ>MXbYEn0Z*V#aARr(hARr(hARr(h"
    "b#7^NUuJJ|V`XGpX<Z;aASfb4QcqAKEFg1aDGCZ8ARr(kAT2E|Eg)%hWo;mAZ)a(9bZKLAIv{9ocOY$HZh0VTZ(?&GZe?X;AZ1}="
    "Xdr2HWo;}VXm58QZFOU4AZc?TV_|Y~X=P*}EiElA3LqdLAZ}%4WM67;VsjupAbWiZARr(hW^ZyJYHwm7X>K5DZ(?&g3LqdLARr(h"
    "AZcbGYHwmLZe?X;AZc?TZf|rTPH%2yItm~lARr(hARr(hARumKWn^D!Z(?&>YHwmLZe?X;T_8OmZe?X;Uuth+b1r9PbSP?XVlHlF"
    "Wn?TMFexA_ATbIcARr)PVRCY5Wn>^dAbWiZARr(hV_|Y~X=QSAAUz;^eF`8TARuOMav*6eAZd0WX>K58Zgp*Ca$$63C~0nXb16Cs"
    "ARr(hARr(hW^ZyJX>=?gZXjuHAZd0kX>?_6b0{e~3LqdLARr(hARr(hAZcbGZXiA&Fggk#ARr(hARr(hARr(hARr)PVRCY5Wn^1v"
    "bX_1lAY);2a%p8`E@x$QC~0&oATTK)D<EzPARr(hARr(hARr(hARr(hV_|Y~X=QSATWNG%AUz;sVRCY5WpZ;aXJvFKX>=?gFexA_"
    "ATbIGARr(hBOom;EiE8ob!lv5AY)-}WNBn!bY&nYb#7^NEFfxcVksbVV{dY0b094(EiDQlARr)cVQF%6AUz;kT?!x|ARuOMav*6e"
    "AaHMUAZczOWo~tCWpZJ3WhhrJb#7^Nb16CsARr(hARr(hX=Wg4AZczOb#7^NUuJJ|V`XGI3LqdLARr(hARr(hAY*TCbZKsNWeOl5"
    "ARr(hARuXWAUz;yZgz89X<Z5+ARr(hARr)fbZKL2AUz;+bS`vZa%W|9b1r9PbSP;l3LqdLARr(hAZBlJAZjciYHwm7X>K58Zgp*C"
    "a$$63C~9wFb16CsARr(hARr(hARr(hWFS2tWNCABC~$9cEFfxcVlH%PY-K45ARr(hARr(hARr(hWq5RQVIVyqFbW_bARr(hARr(h"
    "ARuF5a&l>9a$jO^Zgq1YJs>a&ARr(hARr(hARr(hX=Wg5Z(=TPWo2X_X>%ZMZ*(9|Z*FBe3LqdLARr(hARr(hARr(hAZcbGX?8AW"
    "WppTNZ(=TPWo2Y6ATTK)J|HkU3LqdLARr(hARr(hARr(hARr(hAY);2a%p99Ut(`=b#ov+AT%`!ARr(hARr(hARr(hARr(hWo&b0"
    "Itm~lARr(hARr(hARr(hARr(hARr?kZe?X;AarjaW@U6^Xdq^CZ*3rRXk}y|W@&PBbP6CKARr(hARr(hARr(hARr(hARuXGAaiJC"
    "WG-iAbSP?XVlHlFWn?TMFexBBJs>bT3LqdLARr(hARr(hARr(hARr(hARr(hAY*TCbZKsNWeOl5ARr(hARr(hARr(hARr(hARu#O"
    "EFffaXdpcxZe?L|Wpi|2b7*B`C~$9cDGDGUARr(hARr(hARr(hARr(hARuLUbaG)JJs@OrXdo*fWNCABD05^iAZl-7E_7*ZWho#n"
    "AY=+4ARr(hARr(hARr)gZ**a7Ut}OXAY>pbAZ2)Ta$yP}ARr(hARr(hARr)VW*}v5WM^S*WjYEVARr(hARr(hARr(hARr(kAZ>MX"
    "bRc47AYo!`Wgv8KAY^YKbZBKDYHwmJAZKNCAYx%-Yan!QAarPDAaiJCWFTQ~WFTa6Z*U4AARr(hARr(hARr(hARr)LEFffJVPk6`"
    "Js@snVRB`2bYF95Wn?I7Z(=TVX>4UF3LqdLARr(hARr(hARr(hAZcbGbZ>NFY+qy`D<EuTZYXMRVlH80bZKvHb15JzAY@`;V{0HQ"
    "ATc05AXhGPbY*aJUu<P&bUF$kARr(hARr(hARr(hARr(hARr)PZ*FvHZgph}ARr(hARr(hARr(hWo&b0Itm~lARr(hARr(hARr(h"
    "ARuXGAarkZVQgPyAS)njWo{^HZ(=TCV{~b6ZgVLhJ|J{;a&B{9Y-MJ2UvzI|VR;}cATc0eZe$>8Z(=TRa%pcMJRmVJFggk#ARr(h"
    "ARr(hARr(hARr(hARr)PZ*FvHZgph}ARr(hARr(hARr(hb7OCEWgtBuYHwm+bZKRBC~9wFE^u;bZz&)uAXG_3QeQ_wP#`NHYHwmL"
    "aB^vHASxhGQb|u=S0F7QbZ>NFY+qy`Dj-)uOiN!wPg7JND<ETGa&l>9a$jO^Zgq1CARr(hARr(hARr(hX=WgEbZKL2AU!=GYHwmL"
    "bZKm5Itm~lARr(hARr(hARr(hARu#NZ*pZID?K1nR7pciUqVk#RZ|KeARr(hARr(hARr)NZet)lAaissVQyn+Z*p@kXJvFKX(<XI"
    "ARr(hARr(hARr)VW*}j1V<2gBAZ~ATAWm;?Wgua0WFTs9VlHcGZe$=KJs=`sZfR{{Y$7@eARr(hARr(hARr(hARr(hb7OCEWgsm*"
    "AVE$;NKaB<S0E}NC}CoAC~9wFE_7*ZWm_;^AT1zaZev?8T`3?dAYo#2C~9wFE_7*ZWm_>_AT1zaZev?9T`4IFARr(hARr(hARr(h"
    "X=Wf~AU!=GFd$)WWFTdDbaG)JJv|^WItm~lARr(hARr(hARr(hARr?kW@&C|b7&xRXk{RGZ*prOZ*CxTXk{RDX>4U6cV!@RbYX5}"
    "Aa8CuAYpDGVPkY@Z*CxHWpZU8V{dbGa|$3JARr(hARr(hARr(hARr?kZf_uWVQgz@Zf7hYb8jGNbRcPSAZ}%0a%_1ZVQhC{d2=9i"
    "Xk{Q`Wpi{Ob#rAPZ)PBLXk{RDb#iVBARr(hARr(hARr(hARr(hb7OCEWgsg(AWu$TR7p%lUqVk#RZ|KeARr(hARr(hARr)fV{dY0"
    "AS*o}HXte>Y-MgJYHwmLVPkY@Z*Frb3LqdLARr(hARr(hAaG%6a&s<WaByXAWGE<eV{dY0EFfttAZjciWq5RQVJRsJARr(haA9e3"
    "b1rjla&#zbWqCbpVQpe$VIXiiAT4lPFkLAM3LqdLAa!nObYEm|Ze<`nAaiAOD0OaWbYEs~a${v=E^B3Zb0{e(3LqdLAZl-7Uvyz>"
    "Wo{rnAaiAOC@BgcARr)aWp`h6VRC0>baNm*AbWiZARr(hW@U6^XkT(=b7gXNWn>^dAbWivARr(hBOqyXWo;lWJ|JUnb#8PZa%FR6"
    "a&~28AZBlJAaH48YjtoSbZBXFAar$dZVDhEARuOMav*bKZ*pZUAZaWhYAhgScyw}MAZczOaA9e3b2<tjARr(hARr)VW*})GX>K5O"
    "ZfSI1WN&U|Aa8OYY9MKDAZl-7Uvyz>Wo|kOARr(hARr(hARr(hV{dMBX>N683LqdLARr(hAZl-7AUz;zZ(?&>YF!E-ARr(hARr)c"
    "Z*w3$AXhGRZfSILTWMViARr(hARr(hX?7qzAZc!Pb6aU$3LqdLARr(hAZl-7Uvyz>Wo|BEWMn96DGDGUARr(hARu*aX>?y?Z*FBS"
    "VPs?|X(<XIARr(hARr)aWp`h6VRC0>baPv2T_8OmYHwmLbZKm53LqdLARr(hAR{1dWo2X_X>?_6KMEiqARr(hARuXGAZl-7E^cLI"
    "WFTpCAZ~ATAWm;?Wgua0WFTpFE@x$QC~9wFE^cLIWGo;sDIh#OATT-#ARr(hARr(hARr(hb7U+aWOHaBJs@snVRB`2bYF95Wn?ID"
    "Z*wUMARr(hARr(hARr(hX=Wf~b7&wvJs>bT3LqdLARr(hARr(hARr(hAR{1fX=7_3b#NeVZ+8kHARr(hARr(hARr(hARr)Nc4293"
    "AUz;+Xk}zBXJvFKYHwmLZe?X;EFdr`AT1zfWpra`Uvg!0WpZ|9WG-iAbSP?XVlHlFWn?TMFewTkARr(hARr(hARr(hARr)VW*}j9"
    "VQFk2JUt*VItm~lARr(hARr(hARr(hARr(hARu*aX>?y?Z*FBSWNC9_VRB?BX(<XIARr(hARr(hARr(hARr(hARr)WZ(?6`VQXb>"
    "E@WwQV_|Y+C~7GRARr(hARr(hARr(hARr(hARr(hV{dMBX>N683LqdLARr(hARr(hARr(hAZs8!ATbIcARr(hARr(hARr(hARr)V"
    "W*};BVlHlFWn>^}ZXhTkS4c%cR3a=OB1T10R7p%pT18SKDLM)uARr(hARr(hARr(hARr(hARr(kAZl-7b0BkcX>4pDZe?X;X>MmA"
    "bZBXFAZc`EZ6I`LVRRsFZy;k~a&l>9WFT{NZ)0m9V{dk4a&rnGARr(hARr(hARr(hARr(hARr)eWo=<;ZfS03AUz;+b!{jyAZBlJ"
    "AZltXAZlVDX>K58Zgp*Ca$$63C~9wFb15KcW*}-}E^cLIWFS2~AZl-7E^cLIWFTQ~WFTs4AZ~ATAZczOYHwm+bYW{{ZYdxuATbIc"
    "ARr(hARr(hARr(hARr(hARr)hZewqDWpZU?AUz;$VR$GpEFf}aZDDC{X>MmAEg)lIa&l>9WG-iAbSP?XVlHlFWn?TMFexbtARr(h"
    "ARr(hARr(hARr(hARr(hYal%!ZDDvQF)Sc$X>KTCc4293EFg7mV{dk4a%E&JAT=o|3LqdLARr(hARr(hARr(hARr(hAY);2a%p8`"
    "TWW7&E^cLIWL+RVAY);2a%p8`E@x$QC~9wFE^cLIWGo;sDIhB#YYHGBARr(hARr(hARr(hARuOCbYo~=a%FR6a&~28TWW7&E^cLI"
    "WL+RVAZBHBV`yJ;Wpib6c4cHPXJvFKYHwmLZe?X;EFdr`AS)ni3LqdLARr(hARr(hARr(hAYo&4X>V?GTWMV&Js?{mP)S2eRZt=<"
    "AZl-7E^cLIWGo<ST?!x|ARr(hARr(hARuLIb7eXTARr(hARr(hARr(hARr(hVPkY@Z*Fs2X<Z;aAX{^EWpH0~Z+BsGWGHZNb1Wcp"
    "WGP(=ARr(hARr(hARr(hV{dMBX>N683LqdLARr(hAZcbGaBp)UJv|_5Z(=TVX>4UW3LqdLARr(hARr(hAYo&4AUz;#X>)WaYHwmL"
    "VPkY@Z*Fs2FkLAiX=Wg4b7^jKbYX5|WhiQIVlH80bZKvHb6YT7EFf%Yb95;nWo&b0AWm;?WeOl5ARr(hARr(hARu8dAUz;zZ(=TC"
    "V{~b6ZgX2OT?!x|ARr(hARr(hARuXGAYm{dJv|^IP)tEXMIt&1ARr(hARr(hARr(hARr(hVPkY@Z*Fs2X<Z;aAX_3(OhH3MA}k<k"
    "Z(=TCV{~b6ZgX2PU0n(wARr(hARr(hARr)RY-wg7VK5*)Js=`bOhHamB035nARr(hARr(hARr(hARr)NV{~b6ZgX2{T_8OmTOv?Q"
    "K~7X6EFfxcVlH80bZKvHb6YW8T?!x|ARr(hARr(hARuLIb7eXTARr(hARr(hARr(hARr(hVPkY@Z*Fs2X<Z;aAX{NDT?!x|ARr(h"
    "ARuLIb7eXTARr(hARr(hARr(hVPkY@Z*Fs2X<Z;aAX{^EWpH0~Z+BsGWGHZNb1WcgZ(=TVX>4UFT?z^yARr)SZ*m}MEFdUrX>MdJ"
    "Aai6XAZczOb#7^NUuJJ|V`XG6X>?_6b0{e~3LqdLARr(hAaHMUAUz;gE_H5cbaPv2T?!x|ARr(hARuXGAaHMUAU!=Gb7UZIav*ST"
    "b0BGMAX7+1L|;@%Ohr>V3LqdLARr(hARr(hAYo&4X>V?GTWMV&Js?{mL{d*sB3%j~ARr(hARr)RY;$Eg3LqdLARr(hARr(hAYo&4"
    "X>V?GTWMV&Js?|ibY*Z~bZ>WIa%3oQZ*wdlb7U!93LqdLARr(hAZ}%MUvyz|XJvGATWMV&Js@*r3JM?~ARr?kX=H3=Aa!nObaOf&"
    "Z*CxTXk{R5VRLjKWMO$AcVTR6Aarjab7*B`J0Nd#Xk~JDX>(;DWO8X{bRcwZcVTj5AaiJCWFTQ6Vrg^=ARr(hW^ZyJX)GXcZ*w4N"
    "ZXjiDb!}yGVRU6ES1xsKX>@ZbItm~lARr(hARuXGAZZ|JZXk7TX>?y?Z*FBe3LqdLARr(hARr(hAY*TCbZKsNWeOl5ARr(hARuXW"
    "AUz;yZgz89X<Z5+ARr(hARr)PZge0$AaiwXD0U!bZ*m}OEFg9uX>K5Cb}ngjWo>gPDIjTPAZs9LZXiuTQcFctUr<3(K}}OB3LqdL"
    "ARr(hAai6aAY^lBAUz;%Wnpq<b97&GXk}z5aBp)d3LqdLARr(hAZcbGV{UXHJ|HkU3LqdLARr(hARr(hAYo&4X>V?GTWMV&Js?{m"
    "L{d*sB3&S9W*}s9Xdpd3ATS_hY;$EGTXS?}a9?z9cVTj5C~$9cEFg1aDP0O6ARr(hARr(hARr)aWp`h6VRC0>baPv2T_8Omb7Tr2"
    "ARr(hARr)RY-wg7WOHaBJ|HnV3LqdLARr(hARr(hAR{1Ta%pCCAYx%-Yan!QcVTj5AarPDAY)~2baG`Nb8jGYXk{R7Wq5QTYHwm7"
    "X>%ZBY;SXA3LqdLARr(hARr(hAYo&4X>V?GTWMV&Js?|ibY*Z~bZ>WIa%3oQZ*wdlb7U!93LqdLARr(hAZ2WGWjYEVARr(hARr(h"
    "ARr)NV{~b6ZgX2{T_8OmTOv?FQ&S>c3LqdLAaissbYXI5Wpr~OJs@sncVBd2a%W|9a|$3JARuyObairWAYo&4X>V?G3JMA%AT2E|"
    "EiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEeazbZDDe2"
    "WpoN7AT2E|EiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElA"
    "Eed31W*}r?X>564WO89?ZYWnQAZc`EZ7DhmARr(hA|fJHZfSILAaG@JAY@^AAarPDAarkcZXj}HZEtpEb0B7NZ*3rSXk{R6VRCC_"
    "bRcGLav*eQX>%ZOa&KgHV{|SeA|eVPARr)QAUz;4AZcbGX>?_6AR#><B1T10R7p%pT18SKAZ2WGWgsvLARr(hWFRX&AaiJMa9?9@"
    "b#8PhS1cfDbY*QRASxgzR8T}9FE1c8DGDGUARuyObairWAY=*(3S?zwAaHVJZE1CFUuJA?Z*nMUbY*QUAXh0m3LqdLAR;0nO=)gv"
    "ZFOxRb75>{AaHVNV`VNNP;zB$X?1NNXK!z0b0BAIb#x$RVRLjKVQyp~bZBKDb7gF7WpW^QXm22GZ+2yK3LqdLAZBTDb95kQWpr~O"
    "bZBKDaB^v5Wh@|bZy<MNAaiAGY#?rBVRCGFAZ2!Ca(Q%UX>MmAX>Dy~WNBe^Wo&sMVQyp~Z*FXP3LqdLAZTxFWFU8FWo{sJXk{R6"
    "VRCC_bRcPSAY^4_aBO)XXKZzJbY)~9K~6*<bZBKDbZ>WVAY^i3X>K5QX>4pDY-wh6AZc_iA|fIRARr(hVqtS-AUz;WK~hUaR9{d*"
    "QbA2qTWNG<ZCzU;VqtS-B3%j~ARr)VW*}ECb97~JAU-`HOhHprUsF^?P#`TJF**t$ARr(hARr)eWps6NZXhuVARr(hWMO$AJs?*u"
    "WMO#<ARr(hX=Wg4bY*QIJv|^IO+`#kP9i!AARr(hARr(ha%FUNa&90o3LqdLAY^i3X>K4rAY@@_Y<XW~a$#w1C|4{XX>?_6DGDGU"
    "ARuXGAZc`EZ6G~8AR<>!PfQ{@3LqdLARr(hAZ8#vATTa9H6UqbAY^i3X>K4sJs>wAWo&b0ATTa6FbW_bARuLIX=Wg4bY*QIJv|^I"
    "O-W2kB035nARr(hARr)SAUz;3E;ThEX=Wf~a$#w1AU-`HHy~wfb7defE;BF+ARr(hWo&b0Iv^k;AaitbVRvF>a&mbJARr(hARr(h"
    "W*|KvFfKJPAZcbGWO89?ZXiBAAU7aoY;$EGFfKDN3LqdLAZcbGWMO$AK0P2ZH986)ARr(hARr(kAZ%%FWnpq4WMyMvc_4IeATS_d"
    "bRcYDb95kcbY*Y~ARr(hARr(hW^!R;AUz-`OhHprUsF^?P#`TJS1xmOWpF7VFCb=YZ((#OOhHprUsF^?P#`TJGBqG7AXHF9DGDGU"
    "ARr(hARuNSJs@TvDj;oPcqlL~Ff1T#X>KSnE-)-0W^!R;DJcpdARr)eWps6NZXj)8cqlO}AZc!NC}LrAWgsdbW+^EO3JPRpW*}{0"
    "a%*LDUvF|`WpZ;Ub95{qS1cf5V{~b6ZgVL*3LqdLAa8PHWpZ;MJs?|M3LqdLAY@^AEFfrab#fp*AXhGAVR<YdS1xF8b#e+IARr)R"
    "Ze(X+ZDk-mAY@^AAU-`HOhHprUqnG!3LqdLAZ>4MWqBYyAXhGJZ*FCI3LqdLAaiJCWFS2tS1xmCWn>B<ARr)aUtw-(ZDDM4AUz;s"
    "Z*^{TUtw-(ZDDM4C|4;8ARr(hZeL?>cXJ>;Aa8eWWn^DrZfR{{Y;!1AEFdC7Pgf!-3LqdLAZ}lCXk}$^AUz;&cWz~5Utw-(ZDDM4"
    "C|4{XB2!33MNlFs3LqdLAR{1SVRL98cV!@Lb#rteYh`6{AZBlJAarkSZ*p>PcPDcoXlZg~b0A@EWFTf`Wn>B<ARr)PVRL9-a%FR6"
    "a&~1PJs@ala%Eq1Z**a7C~awOC^0fDAaissXlZg~b6<C1Zggd2DJdW;AZ}k_ZfR{{Y;zzgAT%`~D<C#73LqdLAZcbGWMO$AJv|^W"
    "Itm~lARr(hARuF5b7)_3Wpib6c4Z(vATSCFARr(hBOom;EiElAEiElAEiElAEiE8(Wo&G7AT2E|EiElAEiElAEiElAEeaqYARr?k"
    "aBN|2Ze?U3W@U18X>4hFWpW^HWo2Y@ASiEcY<VDQZEb08Wo~pJZ*FCCDGDGUARuOCa&%vAWo2X_Js>a&ARr(hW^ZyJD0nO&c_}O)"
    "a&K&9AZczOb963pZ)|0AE@^aSZF49oItm~lARr(hARu&XAUz;gE_7*ZWpi73U0Zlv3LqdLARr(hAZcbGX>(t2Y+-J6D0FNoItm~l"
    "ARr(hARr(hARu99WgtBuWMO$AEg)%bbSQLeE@x$QC?ardVQzF~WM5=qc_J(zWMO$JDGDGUARr(hARr(hARuFMZ*U+zAaralXJvFK"
    "B4cuIa3U!RARr(hARr(hARr(hW_2JvAZc!ND0FNtXJvFKB4%ZBbZKm9dSzr^b#8QNY+qzyc_J(zEiox63LqdLARr(hARr(hAZcbG"
    "V{&hBAU!=GB2!dSL03XWQc_tWAYpD~AShvHWguy8ASgI2AUP}`F)=J4F)}P5F*7V6F*PYEAYpD~AZB$SJRoFYc{&OpARr(hARr(h"
    "ARr(hARr)SWpZ?1Ze?X;AS*o}F$y3cARuOCa&%vAWo2X_Js@pqZYXADa&%vAWo2Y6ATuy23LqdLAZcbGWMO$AK0P2ZH#!O+ARr(h"
    "ARr)SWpZ?1Ze?X;AUz;33LqdLAZcbGZEtR6c_2I>G%z4xZe$>2VR;}tAUHY-ARr(hARr(hW@U18Uv6b(WFS2tFd!fxARr(hARr(h"
    "AR{1SVRL98Wo>10XJu|<c{(6wWo2X_V{dI`b0B7Ea&vSFARr(hW@U18AUz;yZgePfXk}zBXJvFKB1T10R7p%pT18SKEFdr`DGDGU"
    "ARu#PY;0d<AUz;vWpZ>NEg)uPa&%vAWo2XvARr(hX=WgEWo&F;W*|NwFd$)WWFRP4E^u;bV`XzLXJvFKB1T10R7p%pT18SKEFdv3"
    "FexBDJs>eMAa8OYWo~3=VQpm~Z*m}WXk}zybZ>NFY$#VLAU+^CFey3;ARr(hARr(hZ*pX1a&s<WaByXAWGGuAQ$<WnA}k;xMnzIo"
    "NlZyvMN%RxAaiAGY+q(wDGCZ8ARr)jXk}q^AUz;yZgePfXk}zBXJvFKB3DR7K~y3vATTK@3LqdLAY@^Ab6;#_W^^DuAWT71R9{3v"
    "Ss*PSWMO#<ARr(ha%FR6a&~1PJs@shVQy({VQh0CDj;oXZYVP>AZ=lIC@?G_GB_YDAY@^AAS)m-DJdXnW*}r?c_2I>GB_Y*Y;$EG"
    "FbW_bARuXGAY@^AAU-`HGB-L3ARr(hARr(ha%FR6a&~1PJs@shVQy({VQh0CDj;oPcqlL|ATl^0Eg)oJc_1qwF)0clARr)VW*}v5"
    "WM^S*WjYEVARr(hARr)eWpib6c4Z(vATSCbARr)fWo&F;cOX3=cW7l{bRaDta%FR6a&~13ARr(hX=WgEWo&F;cOX6>Fd$)WWFRP4"
    "E^u;bV`XzLXJvFKB3DR7K~y3vATl*6AU-`HGBO}<av)`HWM^S*Wgu^IAaiJCWM6b|bYW~LS1BMqAU8EBItm~lARr(hARupYWMy)5"
    "E@5zRWo~3BTOw0MOiUsyAR<>tML|>|EFg1bY;0e5T`39*ARr(hZDnk4ZXi7%X>N2Vb7*B`E@x$QC?ZWoOixZCEFdr`DGDGUARuXG"
    "AZ=xAZ*CwyATT-#ARr(hARr(hZ*pX1a&s<WaByXAWGGuAQ$<WnA}k;xO+`#kP9iKIZDnk4Ze1w~3LqdLAarkZVQe5hAaiJCWM6b|"
    "bYW~LS1Af0ARr)PVRCY5Wn^D;Z**a7AUz;+b!{kfb!{kVZgePiDIjKVav*CgAa)>WZXjuPE@^aSZF49oAZcbGYanTEAWcD1OGQ*)"
    "P(e~bO;af#W^ZyJX?7rKZXj1KX>N9NDGDGUARr?kWp-t9d30!LZf784VRCY5Wn>_1VQyq|AZczObZBKDb7*B`AYpVMbZBKDWo~33"
    "Z)PBLXk{Q|VR<_scW7aBWp-t9AY^Z4a|$3JARr?kZf|rTW@&UFX>%ZCWpi|LZ+T^8AT2&1W^!+BAarPDAYo>7WpZwBZ*CxOZY&^k"
    "Wo&F9WN&wFAarjaZDDI=AaZYSZ3-YDARusZZ)#;@bY)~9Js@;%bYW~DD<ETGa&l>9WM6b|bYW}?ARr(hX=Wg3Z*_7YK0P2YHXvbc"
    "WFSvgMN&pgPgh@4MNCXO3LqdLARr(hAa8bMa%OCAcOX3=ZDDvQFf1T&a&Ky7V{~O?AT1y`I4KGsARr)RY;$Eg3LqdLARr(hAa8bM"
    "a%OCAcOX3=ZDDvQFf1T+Z**a7AT1y`FewTkARr(kAYp8BWnpA_EpTjMZf<2{Aa`hGVRSEMWpZ?BY-xIBav*bIY-MvGa%E(7V`U(8"
    "Xk{R8c4cyAY;Si8ARr(hW^ZyJZy;%IAa8PHWpZ;m3LqdLARr(hAa8bMa%OCAcOWf2AZc!NC~sRbT`3A6ARr)bc4cyAY;SiUJs@pi"
    "cqlL|Aa8bMa%OCAcPR=WARr(kAY*7{VQ^)0bS-9Sa&vSbZ*pX1av)}Jav)}Ja${v=Aah}CWpfH3ARr)ca%F8GJs?{mO-W2kA}k;x"
    "S5Hq&A}k;xQ&dtxS3*TnQduG_AR<LaM<OgBB11t^QcqMOEFdCOPfbBoPa<6kARr(haB^jBE^}{kbSP_Oc|B}lZDM3$AZc_uAXhGM"
    "a%p2_b1r9PbSP<bEFeumQcFctUr<3(K}}OzX>?s%B4S~4Wg=ZEDGDGUARuOMav*7RWo;m7ZXj@SWo<eNARr(hARr(hZXi7%X>N2V"
    "b7*B`E@x$QC~0(MZ7d)#DJcpdARr(hARr)VW*}}LJUt*VItm~lARr(hARr(hARuFJZggpGb!7@5ARr(hARr)VZgwC&AXhGFZgwtb"
    "WppTMbY*QUAWcD1OGQ*)NiZo2ARr(hARr(hX=Wg4bY*QIX>K4WB1K0>A}k;xLqSqfPgEi-AR<&xO+i#oA}KlwARr(hARr(hARr(h"
    "W^8Y7a$j&DJs>e4X=Wg0Ze(X+ZDk;3Y;$EGX>N2VO+iviMO0r<K~h0YQ(I|tWo=zsB4S~4Wg=Z5Dj+Z}Hz^7rARr(hARr)RY;$Eg"
    "3LqdLARr(hARr(hAZBcDZ*pI7AUz;(a%F94b!}f}Y;SLJC~0(MZ7d*HDGDGUARr(hARucXJs@>%X>@a6VPbD~WnXY|X=7z5X>?_6"
    "EFfuab}S%fY;SLJUvMlSZYc^NARr(hARr)VW*}2VOiW)$PgPQ1Pfko(AYpD~AZc`EZ6IlGASfbDNlZ&3EFdCRPftuDEFdCNR8m1#"
    "LPb(iSt2PQVQyp~Zf|rTWo~3=VQpm~VQyp~WMO$AJRmYRItm~lARr(hARr(hARr?kb7gF7AaHVJZE1CFAZKrHWOE>9X>xOPAarPH"
    "Zf78AZXk4MWgu;Da&BpEXCNqIWoB=3Wgv8DWgv8UaA{*<YziPCARr(hARr(hARr?kZ*XvLZe?zCCvzZZZ*_7lF(71hZE!3gVP<q?"
    "av*eQWgu^MWpZw5XJ~XFbZ>WVAY^i3X>KVjAZ~ATAY^rNX>Ml<ARr(hARr(hARr(hBOr8WWgui>c_3+SbZ;PGAY^55a%FRKWn>_2"
    "VRCC_bP6CKARr(hARr(hARuXGAZTxOav(k+Q$<WnUr0|?QeR9#Q&c(%ARr(hARr(hARr(hARr(hYal%!FbW_bARr(hARuXGAa8bM"
    "a%OCAcOX6>Fd$)WWFTuGJRoj53LqdLARr(hARr(hAZ2)Ta$z7nAZ=-GC~hDvAZsijZ+2yJW^8YFDGDGUARr(hARr(hARucXD?K1("
    "cyw}M3LqdLARr(hARr(hAa8bMa%OCAcOWf2AZ2)Ta$yP}ARr(hARr)VW*}v5WM^S*WjYEVARr(hARr(hARr)XAUz;%3LqdLARr(h"
    "AZcbGYal)#Fggk#ARr(hARr(hARr)ba%5$4b1q?UaAj^}C|e>^MNCX0EFfugWo;}VYh5V{ARr(hX=WgAc4cyAY;SiUJ|HkU3LqdL"
    "ARr(hAR{1ibZKmCAarkUAZ>MHXgVNdb!~7UcW7l{bRcqNb7gXNWgssgW@U18X>4hFWpW^LWpib6c4Z)RZ*K}9ARr(hARr)jUu<P&"
    "bRaz-X>N2Vb7*B`E@x$QC?Z!#ML|>|EFdr`DIhH%b9HSfX>N2VZ(A~5DIjKVav*OYX>K5Ia%5$4b0BGEAa7eST_8O@AR<>tML|>|"
    "DGDGUARr(hARuXGAa`GEWoC3BJ|HkU3LqdLARr(hARr(hAZs8!AZ=-GD0g3MWoC3NAa8bMa%OCAcPR=WARr(hARr(hARr)ba%5$4"
    "b1q?UaAj^}C|e>^MNCX0EFdCRNJT+ZA}k<lT`3A6ARr(hARr(hARr)bc4cyAY;SiUEj=J>3LqdLARr(hAZA}|WoC3BJs@drbSQIZ"
    "Wn?aAWppSaMnzIoNlZyvMN%RxATTK@AT1zsb!{kVZgePbTQXfKAZBlJAa5XPZXj=RWMy)5AZcbGZ(A{4AU!=GB1T10R7p%pT18SK"
    "DGDGUARr(hARuXGAa8bMa%OCAcOX6>Fd$)WWFTf=Y-MJ2AU+^4Itm~lARr(hARr(hARupYWMy)5E@5zRWo~3BTOw0MOiUsyAR<OZ"
    "QdCJyNm@lxA}k<nX>KTHUu<P&bSxlmc4cyAY;SidT`3A6ARr(kAZ2rOX>DP2Wn>_5a&Kd0Wn^<8C}VGKb7gXNVRUJBWho$ZZy;uM"
    "Ze$>Eb#h~9VRL13AarPHb0Bnea&8JBARr)ca&Kd0Wn^<8Js>bHFbW_bARuOMav*OYX>K5Ia%5$4b2<tjARr(hARr)VW*~1{FkK)$"
    "Js=`eMNCX0Itm~lARr(hARr(hARuXOb|5_<S1xI8b}naSbSQ6IF<mSmO+iviMO0r&FewTkARr(hARr(hARr)gZ*(9%ATSCbARr(h"
    "ARr(hARr)SZ*m}CAZczOa$#;~WhiNGbSQ6IGF>StItm~lARr(hARr(hARr(hARusZAUz;$VRCC_bYF0CX=7z5Z(A{4EFfuab}0%V"
    "ARr(hARr(hARr(hARr)gZ*(9lJs@y$3LqdLARr(hARr(hARr(hAZcbGaB?6%ATc@$ARr(hARr(hARr(hARr(hARr(hX>N8PD?K1F"
    "3LqdLARr(hARr(hAaHVTV`XJzb08}{AarkZ3LqdLAY);3XdpcxZEtR6c_1qwFfKVDDj;xjZ)0U;WOE7%ARr(hBOom;EiElAEiElA"
    "EiElAEiE8uX>w(AAT2E|EiElAEiElAEiElAEeaqYARuXGAZ~ATAZ2c3XJKt+Aa8OYXm53LAUr)FGCB$%ARr(hARr)jVQzFFJs@**"
    "E@)|TWpiJ5VQzF~WC|c4ARr(hARuXGAZ2c3XJKt+Itm~lARr(hARr(hARu>PZge0$AZ=-GD0g9QbSxk^DGDGUARr(hARuUAc4Z(v"
    "AXhGEX>w(AUvzI|VR;H5ARr(hARr)aUubD^WgtBuZDDvQFf1T<VQzFFEg)!Nc4a9FARr(hARr(hBOq^XY<VDPX>w&CX>K5NXk{Q~"
    "X>xOPAZTxOa&sVWW*~HEWgui>c?uvPARr(hARuXGAZTxOav(k+Gdc<&ARr(hARr(hARr)aUubD^WgtBuFbW_bARr(hARuFJb95j*"
    "ATSCbARr(hARr)XAUz;33LqdLARr(hAa`hKY-J#8AUq&$UubD^Wgua0WFTy1ZYXbZWMy)5DIh!`F)%s`ARr(hARr(hARr(hV<0^s"
    "W@%zyV{dbGC}?4JWgsgcYbgpKARr(hARr(hARr)VW*}o>b7&wfAY*THbRaDtV<0>rFggk#ARr(hARr(hARr(hARr)Oa%Ev_3LqdL"
    "ARr(hARr(hAY*THbRa7|AY%$3ARr(hARr(hARr)XAS*o}F$y3cARr(hARuOMav)zIX>K5LVQyz-C~GM?3LqdLARr(hARr(hAa8PH"
    "WpZ;aVQ_F|Ze%E1B1lP6MIv1(3LqdLARr(hAY);3Xdo>;AY*THbP5U}ARr(kAT2E|EiElAEiElAEiElAEg)=RZe$=WEiElAEiElA"
    "EiElAEiEkyARr(hZeMk7Y#==#Y-MgJS1xsKY;R+0Wn?J|ARr(hX=Wg9Uv+M5AUq&5AYpD~AZ~ATAZ2c3XJKt+AYpD~AY@^AAUr)F"
    "F*YD!Ze$>AWo{^Ma%5$4b15J^ATcmH3LqdLARr(hAaHVNV`U&cAWT6{L|;%+Nkc_bTW()<ZfqbeATeDEARr(hARr(hZ)+euAVy(q"
    "b7cx3ARr(hARr)PZ+Bm0WoBt(X>=exAZ=lIC@?G_b963aZ+BmGVRC0>bRaDtZeL?>cXKHUARr(hARr(hX=Wg9Uv+M5AU!=GF(6@X"
    "WFTZ=c_2PLAT%IhZe$>1VRL98Eg*1mX=7y|K0P2~VRL9-a%FR6a&~1p3LqdLARr(hARr(hAa83RJs?zab!7@5ARr(hARr)VW*}}~"
    "b#81RJv|^YAYpD~AY@^AAU-`HI3QteWFTW<b7&wfAaHVNV`U&dJs@LYb7)_3Wpib6c4Z(dAT%&AASxhZZ+Bm0WoBt(X>=efATuy9"
    "Itm~lARr(hARr(hARupRAUz;da&=`2ARr(hARr(hX=WgAYdQ)bARr(hARr(hARr)ba%5$4b1q?UaAj^}C|e>zRasw5K~6*>T`3A6"
    "ARr(hARr(hARr)PVRL98Ej=J`a%p2_3JM?~ARr?kEiElAEiElAEiElAEiElAAaiA9WOE=bEiElAEiElAEiElAEiEkyARr(hX=Wg9"
    "Z*(AKZe(X+ZDk-~Ze$>AWo{^Ma%5$4b15J^ATcmH3LqdLARr(hAa`MIbRaz-dwmKZARr(hARr)SZ*m|gcq|}!DJ&pzZ){~CX>K5M"
    "bS`plY-MvUX>?_6b0{e~3LqdLARr(hARr(hAaraXJs?*ubZKm5b6a^`TX<axARr(hARr(hARr(hX=WgFY#?cKAWm;?Wgu^IAZc@7"
    "cV%T{D0FNoItm~lARr(hARr(hARr(hARuXGAaZYPWgtC0AR<jgOixZCAYpD~AY@^AAU!=Gb963kWo&P5UvO+;ZggK{VR<?VARr(h"
    "ARr(hARr(hARr(hARr(hcVTXHTOv(GOixZCT_8OmcVTXHE@x$QC?ZWoOixZCEFdr`AS)m-3LqdLARr(hARr(hARr(hAZ2W6W*~BJ"
    "Y-J!lJs=`eR8m1#A|PRIWFTZ=c_2PLATuCgZe$>2VR;}tJs?w5QbAW=OhHprUr<azPE<MyARr(hARr(hARr(hARr(hARr(hcVTXH"
    "TOw0bQbAWjMN(2(B3&RoAa`MIbS`IQbSNTIR8m1#LPb(iSt2YTFexA_ATbIcARr(hARr(hARr(hARr)RY-wg7a&K&9AU!=GB11t^"
    "QcqMOAYpD~AY@^AAUr)FLqSqfPgGw_K~q#;P)tEiR5}VEARr(hARr(hARr(hARr(hARr)jVQzF=B11t^QcqMOT_8OmcVTXHE@x$Q"
    "C?Z2aQc_P;A}k;<DIhB#F$y3cARr(hARr(hARr(hARuLIX=WgDZ){~CX>K4WB3DR7K~y3vAR<##QbAWDEFdCHMNCglA}k;xLqSqf"
    "PgEi)AYpD~AY@^AAU-`HF(6@XWFTZ=c_2JJAXi95K~!H%K~q#;P)tEiR5}VEARr(hARr(hARr(hARr(hARr)jVQzF=B3DR7K~y4L"
    "AUz;=VQzFTXJvFKB3DR7K~y3vATTK)D<ClnARr(hARr(hARr(hARr(hWo&6?AaZYPWgtC0AR<9dNligaA|PRIWFRPHVR;}tJs>e4"
    "Z*m}HVR;}vJs>eSDIj5PWFTZ=c_2JJAXi95K~!H%K~q#;P)tEiR3Kq)WFT&DbRc(OZggL5Z*pZ{b98cbV{~<LWpgNVbSxlODLM)u"
    "ARr(hARr(hARr(hARr(hARr)jVQzF=B3DR7K~y4LAUz;=VQzFTXJvFKB3DR7K~y3vATTK)D<ClnARr(hARr(hBOq^XY<VDJb$K9W"
    "Z*m}aXkm09V_|L}Vr3w3Y+-J6Wn>_9Z)9P4ASi5MVsCOFVsCYBWGNskAaiYDY-}K6b!KK|ata_IARr(hARujFcwcaAVQzF^bZ=x~"
    "c_2L?ZDDvQGAtl$Wo{@}E_H5cbaN>nDj+f`3LqdLARr(hAZBlJAY*cGa3E=JASfbDMNCglA}k;xQ&dtxS3*TnQduG_AR<FSQc_P;"
    "A}k;xS4c%cR3a%l3LqdLARr(hARr(hAZ{Q%Aa`MIbS`IQbSPtTZ*VLiFexA{AZc!NC|53XWo2Y@E@x$QC}VPOa4aA&DJcpdARr(h"
    "ARr(hARr)VW*}p7Z*U+zJs=`eR8m1#LPb(iSt2?LARr(hARr(hARr(hARr(hZXi7%ZE0>OZY&@)FewTkARr(hARr(hARr)RY-wg7"
    "V{&hBAU!=GB3DR7K~y3-3LqdLARr(hARr(hARr(hAZ{Q%AZ=-GC~hnuZDDv{aBN|2bYFCDWMO$J3LqdLARr(hARr(hAZcbGZXi58"
    "ATT-#ARr(hARr(hARr(hARr(hV{dMBX>N683LqdLARr(hARr(hAY&jsAVX45P*YoDa&K^5TOxC1Wn>~<3LqdLARr(hARr(hAaZ4M"
    "WpZ|9AUz;sVRL9-a%FR6a&~1PX=Wf}a&K@TZf|rTX>K4WB3DR7K~y3vAR<FSQc_P;A}JtcY;$EGZE0>OV_|b>Uvg!0WpZ|9EFdv5"
    "FewTkARr(hARr(hARr)XAUz;yZgePZVR$GoEFdUjVRL98Eg*7bb7gXNWho#pFCb$nDGDGUARr(hARr(hARucXJs@pqZYXOkAZ{rN"
    "ARr(hARr(hARr(hX=Wg6AU+^4Itm~lARr(hARr(hARr(hARupYWMy)5E@5zRWo~3BTOvYLSzl8{MMNSjAY*cGa4aBeT`3A6ARr(h"
    "ARr(hARr(hARr)PVRL98Ej=J>ASxhZ3JM?~ARr?kEiElAEiElAEiElAEiElAAYpE4ZDDM4AT2E|EiElAEiElAEiElAEeaqYARuXG"
    "AZ~ATAZ2c3XJKt+AYpD~AY@^AAUr)FQ%FTcP+v?zQ&e9<RaqcmZe$>6Z*_7YJUt*bAYpD~AZ%rBC~tCPWpZ;VAUq&3Fggk#ARr(h"
    "ARr(kAZTxQAZ=l8c_46Mb98lbWpf~9cxiKVATJ<eVQwI3Z*z1CARr(hARr(hWo>YDd0%j0b98lbWpf}sATSCbARr(hARr)RZE$pX"
    "Utw-(ZDDL*bZKm5b09q+FbW_bARr(hARuOMav&&pEFgI)EFf}kY-J#6ZXk1XE^=>dWpgfRbY*RGC@DG$ARr(hARr(hARr(hbZj6!"
    "AXhGQX>4V4TX|hucwGu0ARr(hARr(hARr)VW*}*EUuA7@ba`KMbaHiLbSQLeDIj5PWFT~GE@x$QC?acVZe$`UAU!=GB2Yn7R8>+%"
    "B035nARr(hARr(hARr(hARr)RZE$pXUvOb_bairNb08}{ATbIcARr(hARr(hARr)VW*~BJY-J!lJs=`MPDxEcOd=p*Ze$>IY#?cK"
    "AWm;?WjYEVARr(hARr(hARr(hARr)RZE$pXUtw-(ZDDL*bZKm5b08}{ATbIcARr(hARr)PVQ^t%X>@rYJs@RmaCCWJaA9+Fb#i5M"
    "AS)ncZE$pXUtw-(ZDDL*bZKm5a|$3JARr(hARu*aaBN{?Wn>^dASiBMV{dnJAT1zcZ*^{TUtw-(ZDDM4C|4{XB12DCA}J{#D<CLt"
    "Uvp?>WpE%ZAY*TJZggK^ZfR{{Y;!1AEFdCNNJT|ZA}J{fARr(hARr(hV_|S%V`+4GAT2#0b#8EMVPj=v3LqdLARr(hAYyfSUt=IW"
    "AZ=lIC@?G_b963aZ+BmGVRC0>bRaDtZeL?>cXKHUARr(hARr(hVs&|6b09q+ZDDvQFf1T*bS`sfWo2+*bYXI5Wpp4dAZ}lCXk}$^"
    "DGDGUARr(hARr?kVs&{SY-w$2bY)~9VtF8AVRL98CLm*BaA9L<ba^ZwaB^j4WpW^6Z+CMbWnpq`c?uvPARr(hARupYWMy(+Y-w|J"
    "AUz;kT?!x|ARr(hARuOMav*DIZe%PVV{UXHX>K4WC?Z2oS0XGRVs&|6V<{{kC?ZoxMMY2|EFfZad0%rWDLM)uARr(hARr(hARr)X"
    "AUz;33LqdLARr(hARr(hAa`hKY-J#8AUq&rZge1FZe$>1VQ^t%X>@rYJ|HkU3LqdLARr(hARr(hARr(hAY&jsAVE$^O+idkTWe`<"
    "WL;Y#V{dbGB3%j~ARr(hARr(hARr(hARr)VW*}o>b7&wfAY&jrAY);3XkT(=b7gXNWjYEVARr(hARr(hARr(hARr(hARr)Oa%Ev_"
    "3LqdLARr(hARr(hARr(hAY);3Xdo>;AY%$3ARr(hARr(hARr(hARr)PVQ^t%X>@rYEj=JH3LqdLARr(hARr(hARr(hAZs8iJs>d("
    "ARr(hARr(hARr(hX=Wg6AU+^4Itm~lARr(hARr(hARr(hARupYWMy(+Y-w|JE@5zRWo~3BTOvYLSzkd;NligaA}k<lX>MdJAZuMI"
    "3LqdLARr(hAa8PHWpZ;aWq5RDZe%ELa%5$4Uu<b}bSVl7ARr(hBOom;EiElAEiElAEiElAEiE8sWo2X_cW7l{bRaD)EiElAEiElA"
    "EiElAEiDQlARr)VW*}~FbRcDJWM^S*Wgua0WFTy1ZYXbZWMy)5DIh!`F)%s`ARr(hARr(hcW7l{bYE_7cOX3=X>N2Vb7*B`E@x$Q"
    "C?Z!#ML|>|EFdr`DGDGUARr(hARr?kcW7l{bRc74a&l>9WFTUBAa!nObaNngX>4pDW@Tk$AarkJVR<_sbZ>2Ma&m8XAZ}%4WOE>H"
    "Utw-(ZDDM43LqdLARr(hAZ}%4WM6b|WMO$AJs>a&ARr(hARr(hW^ZyJD0nO&c_}O)a&K&9AZczOb963pZ)|0AE@^aSZF49oItm~l"
    "ARr(hARr(hARu&XAUz;gE_7*ZWpi73U0Zlv3LqdLARr(hARr(hAZcbGXkl|-VQy({VQeULY$+gNZe$>CZ*(AZY%XVIbSNTbWn^D;"
    "Z)9P4A}k<AVQh0{DLM)uARr(hARr(hARr(hARr)aWo2YvbZ=x~c_1r2ATbIcARr(hARr)PVRCY5Wn>^dAaiwXC~0nVC~0;sXJvFK"
    "B3DR7K~y3vATTK@AZBlJAZd0WX>K4_E@^Idb14cSARr(hARr)fXm4_KAUz;33LqdLARr(hAZcbGXm53LAUr)FGB7#{ARr(hARr(h"
    "ARr(hb7*gJbRaz-Ze?X;UvzI|VR;}eAY);2a%p8`AT1zwXk}q^Uv6)A3LqdLARr(hAZ2WGWjYEVARr(hARr(hARr)fXm4_KAUz;%"
    "Utw-(ZDDM4AT1zwXk}q^Uv6)AAT1zcVRCY5Wn>B<ARr(hARr)VW*}r?c_2PLATl^Q3LqdLARr(hARr(hAaiJMa&#a)AZ=-GD066U"
    "a&#;pFewTkARr(hARr)VW*~EDZ*p`XJ|HkU3LqdLARr(hARr(hAaHUZJs?*uaB^v5WpgfPWppSaS4c%cR3a=OGBqh6D<CrpARr(h"
    "ARr(hARr(hYal%!X>N2VZE0>Ob7*gJbSxlkVR$GoEFdUjVRL98Eg&^1ATKW<aB?XrDGDGUARr(hARr(hARuXGAZs8#ATT-#ARr(h"
    "ARr(hARr(hARr(hZ*pX1a&s<WaByXAWGGuALRDE`P*P7sRYO!FEFdCRNJT+ZA}k<lT`3A6ARr(hARr(hARr(hARr)PVRL98Ej=J>"
    "ASxhmataC{ARr(kAT2E|EiElAEiElAEiElAEg)oJc_1(#b8uy2X<=+2Z*pX1a%pa7AT2E|EiElAEiElAEiElAEeaqYARuXGAXhGP"
    "bY*ZLJv|^WItm~lARr(hARupYWMy)5AUz;kTOvqFQbi(NT_7qTPf$fpUr0$(MN=RvAX^F`ARr(hARr(hARr)HB0^PJUqMbuO+ic|"
    "EFdCNNJT|ZA}k<JP(@B(Q%FTcP+cqvARr(hARr(hARr(hTOvYLSzkd;NligaA}k;xLr+&CEFe!%MNVHsPghf2ED9hXARr(hARr(h"
    "ARt>JLRDE`Q$<BYA}k;xO+`#kP9iKIPf$fpUrj|!Pfk-^ED9hXARr(hARr(hARt>JLRDE`Q$<BYA}k;xS4c%cR3a=OPf$fpUsp&)"
    "K~!H;MMXqYT`US9ARr(hARr(hARr)HB0^PJUr<s{L{&pnA}k;xS4c%cR3a=OPf$fpUsp&)K~!HxMMXqiED9hXARr(hARt`|ARr(h"
    "a%FUNa&91Ra%5$4b6YwwFkK1?3L_vbEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElA"
    "EiElAEiElAEiElAEiElAEiEkyBOq;IX>K58Zgg^a3L_vbEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElAEiElA"
    "EiElAEiElAEiElAEiElAEiElAEiElAEiElAEiEkyWMyU`VPkYCZ(?&PItm~lARt#DJs@yla&u)UZ(?&P3LqdLAaiseJs@XgbYF9H"
    "VRU6ES1xdDVR>b8EFf1db97~JDGDGUARuXGAaissaBN|2Uu0o<AR#><S1x2>c{&OpARr(hARr)cY+-I+WMO$Ib95{qS1Af0ARr)R"
    "Y;$Eg3LqdLARr(hAR{1hWoB|^b7&xPZ)|0AAa`hGZXj%7Ze$>JZftL3Yh`30ZE0jJWMO#<ARr(hARr(hX=Wg7Wo{^QbS`plY-MvP"
    "AR#><Y-MgJb#82LV{2t(Uvz0~WpgN3DJeP%ARr(hARr(hARr(hVRLh7XKr6|Z)|0AD06fyAXh00ARr(hYHwn5AUz;rb!lv5Uuth+"
    "b0~9kEFf1Y3LqdLAYo&4X>V?GAUz;qb8~5DZYXnfEFf1bAZl-7b14cSARr)ba%5$4b09q+ZDDe2WprO}a%5$4b0~9kEFf1bAYo&4"
    "X>V?GDGDGUARuO8a&2XDAUz;qV{~b6ZgX2OT_9;@AYo&4X>V?GAZ2WGWguH3P(f2uB3%j~ARr)UVQyq|AUz;qV{~b6ZgX2PI&5Wb"
    "C|53ZZfSILDP0O6ARr)eWps6NZXkOiW?^z|WpW}qAZB55ZDn#SAR=gCZe(*JIv{9aZe(*TAR=vHa%*LDB03;%a%5$4bA1X53S?zw"
    "AYo@^ZgePbVsk7YV{dL|X=go7Z*FBNItm~lARu&dc{&OpARr(hARr)eWps6NZXjV}bSQ6Pb14cSARr)Rcw=R7bRb1|V`Xr3X>V>i"
    "3LqdLARr(hAarthItm~lARr(hARr(hARuO8a&2=UJs?LYZ(?&SAR=aAa&2=WEFfE5DGDGUARr(hARr(hARurcJs@drbSOtCZ(?&S"
    "AR=&VVR>b8A}k;<DJcpdARr(hARr(hARr)UVQyq|AUz;QC}v@DZF5_2T`V9XXkl(-b0RDtTU{v%ARr(hARr(hARr(ha%FUNa&91d"
    "B4%N7ZDn#IIv`sjP(f2uB3&#XB4}Z5WOE`qAX{4^P(f2uB3&S6Z*m}CAZczOXkl(-b6qSTB5h%EYh`pIIv`tJeF`8TARr(hARuLU"
    "V`Xr3AVqj%WpH$9Z*DpYARr(hARr(hARr(ha%FUNa&91dB4%N7ZDn#IIv`sjP(f2uB3&#XB4}Z5WOE`qAX{B5AR=vHa%*LDB03;j"
    "U404)Ute}*a&u{KZeL#@Js=`jH90gba(QnpY-x67EpKpTZfS03A_@"
)

output_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd() / "x594_output"
output_dir.mkdir(parents=True, exist_ok=True)
main_path = output_dir / "main.py"
main_path.write_bytes(base64.b85decode(SOURCE_B85.encode("ascii")))
actual_sha256 = hashlib.sha256(main_path.read_bytes()).hexdigest()
assert actual_sha256 == EXPECTED_SHA256, (actual_sha256, EXPECTED_SHA256)
print("Wrote exact X594 entrypoint:", main_path)
print("SHA-256:", actual_sha256)


In [ ]:
from IPython.display import Code, Markdown, display

display(Markdown("The exact emitted root `main.py` is shown below."))
display(Code(filename=str(main_path), language="python"))


In [ ]:
import hashlib
import importlib.util
import py_compile

py_compile.compile(str(main_path), doraise=True)
spec = importlib.util.spec_from_file_location("x594_emitted", main_path)
module = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(module)
assert callable(module.agent)
assert hashlib.sha256(main_path.read_bytes()).hexdigest() == EXPECTED_SHA256
print("X594 standalone entrypoint: PASS")


In [ ]:
from kaggle_environments import make

smoke_results = []
for x594_seat in (0, 1):
    agents = [str(main_path), "starter"] if x594_seat == 0 else ["starter", str(main_path)]
    env = make(
        "kaggriculture",
        configuration={"episodeSteps": 720, "seed": 594594},
        debug=True,
    )
    env.run(agents)
    final = [(state.status, state.reward) for state in env.steps[-1]]
    assert all(status == "DONE" for status, _ in final), final
    smoke_results.append({"x594_seat": x594_seat, "final": final})

print("Official Kaggriculture smoke: PASS")
print(smoke_results)


## Reuse checklist

1. Download the notebook output `main.py` from the root output directory.
2. Verify that its SHA-256 is `d946545e01d5a07485ca348c16080b61ecd4f2eff6a8c5fc08be7a449c43abc5` before any competition submission.
3. Keep this notebook publication separate from competition submission decisions.
